# Debenture liquidity provider: from raw data to results, in one notebook

This notebook is **self-contained**. It downloads every dataset from its public source, cleans it, builds every feature, trains the model walk-forward, simulates the fund, tests robustness, measures capacity and gives the live tools. It imports nothing from other project files.

**The strategy in one paragraph.**
- Sometimes a seller **dumps** a debenture: a trade of R$ 300k or more at a price at least 50 bps below the bond's fair value (yesterday's ANBIMA mark).
- We are the **buyer** of that trade. A model, trained only on the past, decides which dumps to buy.
- We sell at the next real trade after 21 business days. For IPCA+ and pre-fixed bonds, the rates risk is hedged with Tesouro. Cash we are not using earns CDI.
- The edge is the price concession a forced seller pays to whoever takes the other side.

**Run time.** About 20-40 minutes from the local cache. A first run from an empty cache downloads about 3 GB and takes several hours; most of that is CVM fund files and the per-bond ANBIMA history.

**Contents**
0. Glossary
1. The strategy, step by step, and why it should make money
2. Everything we tested before arriving here (and the look-ahead traps)
3. Setup and configuration
4. Data collection (each source: what, where, how, point-in-time rule)
5. Cleaning: marks, durations, payment events, curves, credit-fund stress
6. Building the dataset: dumps, outcomes, rates hedge, features
7. The model (walk-forward LightGBM + calibration)
8. The fund simulation (book)
9. Backtest and equity curve vs benchmarks
10. Robustness analysis
11. Scaling
12. Live test: data to request, daily routine, go / no-go criteria
13. Live tools: final model, bid sheet, RFQ scorer, paper-trade log

## 0. Glossary

| Term | Meaning |
|---|---|
| **Debenture** | Brazilian corporate bond. Most trade over the counter and are registered at B3. |
| **PU** | *Preço unitário*: price of one debenture in R$. |
| **PU par** | Value of one debenture at its contractual rate: notional plus accrued interest. It grows every day and drops when a coupon or amortization is paid. |
| **%PU par** | PU / PU par × 100. 100 = priced exactly at the issue rate; 98 = 2% cheaper. Comparing prices in %PU par removes accrual and payments, so prices on different days are comparable. |
| **Indexer** | How the bond pays: **CDI+** (CDI plus a spread), **%CDI** (a percentage of CDI), **IPCA+** (inflation plus a real rate), **PRE** (fixed rate). |
| **CDI** | Brazilian overnight interbank rate. The benchmark for cash and for credit funds. |
| **ANBIMA mark** | Daily indicative price ANBIMA publishes for each debenture (*taxa indicativa* → PU). Mostly a 3-day average of dealer quotes plus a little weight on trades, so it lags the market by about 3 days. |
| **SND** | *Sistema Nacional de Debêntures* (debentures.com.br): public registry with each bond's characteristics and payment events. |
| **Dump** | A trade ≥ R$ 300k whose %PU par is 50-2000 bps below yesterday's ANBIMA %PU par. |
| **bd** | Business days. |
| **Capture** | Share of dumps that actually reach our desk (a seller has to call us). Assumed 20%. |
| **Walk-forward** | Each quarter the model is trained only on trades whose result was already known before that quarter. |
| **Duration** | Price sensitivity to a 1-point change in yield, in years. Used to size the rates hedge. |
| **ADV** | Average daily traded volume (R$) of a bond over the past 63 bd. |
| **CVM** | Brazilian securities regulator. Publishes every fund's daily NAV and flows (*informe diário*) and monthly holdings (*CDA*). |
| **Credit-fund stress episode** | CDI credit funds lose to CDI over 21 bd **and** have net redemptions over 21 bd. Forced selling follows. |

## 1. The strategy, step by step

**Why it should make money.**
- Debentures are illiquid: a given bond may trade a few times a week.
- A seller who must sell **now** accepts a price below fair value to get out. Examples: a fund paying redemptions, a risk limit being breached, a dealer that will not carry inventory overnight.
- Whoever takes the other side earns that concession when the price drifts back to fair. This is what dealers live on.
- We measured it: buying the **next day** after a cheap print loses money (−15 bps per trade). The edge only exists for the **counterparty** of the dump itself. So the strategy has to be a liquidity provider that answers calls (RFQs), not a screen that reacts to yesterday's prints.

**Step by step.**
1. **Fair value** of each bond = yesterday's ANBIMA %PU par, which is known before any of today's trades.
2. **Dump** = a trade ≥ R$ 300k whose %PU par is 50-2000 bps below that fair value.
3. **Model.** A LightGBM classifier estimates P(this dump earns money over 21 bd, rates-hedged, after costs).
   - Training: each quarter, only on dumps whose exit had already happened.
   - Calibration: isotonic regression on earlier out-of-sample quarters.
   - Features: discount, size, size / ADV, duration, price level, mark momentum (bond and issuer), the issuer's worst bond, days since the last trade, time of day, how many dumps the bond had earlier that day, round lot, SND payment failures, days since a credit-fund stress episode began.
4. **Decision.** Buy if P ≥ 0.70 and neither the bond nor the issuer failed a payment in the last 90 days.
5. **Size** = the smallest of:
   - the dump size;
   - 4% of capital per bond (1% if the bond or the issuer's worst bond is < 90% of par);
   - 15% per issuer;
   - free capital.

   Only 20% of dumps are assumed to reach us, drawn at random. Results are averaged over many draws.
6. **Exit.** Sell at the daily VWAP of the first trade day after t + 21 bd. Windows containing a scheduled cash event (coupon, amortization) are skipped, so payments never contaminate the return.
7. **Hedge.** IPCA+ and PRE positions carry a short, duration-matched Tesouro position. It is sized by a **walk-forward beta** (how much of the Tesouro move the debentures actually followed over the previous 12 months).
8. **Cash** not in positions earns CDI. Cost: 1 bp per side, the desk's figure. The measured dealer markup is in § 2.

**P&L of one trade:**
- **Bond leg:** PU_exit / PU_entry − CDI accrual over the holding period − 2 × cost.
- **Hedge (IPCA+ / PRE):** subtract β × (Tesouro return − CDI).

The PU change already contains par accrual and the change in %PU par.

## 2. Everything we tested before arriving here

Every line below was actually run on the same data, with the same honest execution rules. The final strategy is what survived. **None of this research code is needed to run the notebook.** This section is the lab record, so you know what was tried and why it was dropped.

### 2.1 Costs and fair value
| Model | What it does | Result | Verdict |
|---|---|---|---|
| Dealer back-to-back pairs | Two trades with the same quantity within minutes = a dealer passing a bond through; markup = price difference | Round-trip markup by size: < R$ 100k ≈ 178 bps; 100-300k ≈ 146; **300k-1 mi ≈ 60 (median 1)**; 1-3 mi ≈ 20; 3-10 mi ≈ 10; > 10 mi ≈ 7 | Basis of the measured cost table |
| Pair initiator | Which leg came first = who started the trade | Only the initiator carries flow information | Used in research only |
| Kalman fair spread | Student-t Kalman filter on prints + marks | Works as a fair value **only with the state before the print**. Using the posterior gave fake 83-90% precision | Replaced here by yesterday's ANBIMA mark (simpler, same result) |
| Issuer-curve relative value | Cheap vs the issuer's own curve | +6 to +12 bps per 21 d when trading anyway | Overlay, not a strategy |
| Execution models (Hawkes, optimal stopping, Almgren-Chriss) | Wait for a cheap print, with a deadline | +7 to +16 bps per purchase | Execution rule only |
| Default hazard / Merton distance-to-default / fire-sale network | Predict drops from fundamentals and common holders | Monthly drop AUC ≈ 0.75, not tradeable on its own | Filters only |

### 2.2 Predicting price direction (all dead or artifacts)
| Model | Result | Verdict |
|---|---|---|
| Issuer/indexer-neutral 21 d return model | IC 0.075; avoid-the-worst rule +0.6-0.8%/yr | Overlay only |
| Daily mark-drop model (5-40 d) | 70-79% precision the same day, **45-54% with a 1-day lag** | **Artifact.** The ANBIMA mark needs ~3 days to absorb news, so the model only forecast the mark catching up with trades. |
| Credit-spread drop model v1-v3 | Same lag problem. As a tail-insurance sell signal: +0.7% NAV / yr | Insurance, not alpha |
| Rise model (target CDI + 10%) | Hit rate inflated by "lucky print" exits | Dead |
| Inflation vs Focus survey | No edge | Dead |
| Buy in the primary offer | −22 to −31 bps vs CDI | Avoid the primary |
| Cross-sectional LightGBM, all features | IC 0.03-0.05, loses after cost | Dead |
| Issuer equity residual leads credit | t = 2.7 at 10 d, 8-30 bps | Real, smaller than cost |
| Issuer's most-traded bond leads its other bonds | +61 bps gross, negative net | Dead |
| Distress events (adviser hired, CFO exit, equity crash, rating) | Net ≈ −cost | Dead |
| Issuer option implied vol (B3) | −31 bps / 20 d (t −1.9) | Weak |
| Listed FI-Infra funds' premium to NAV | No lead | Dead |

### 2.3 Liquidity provision: the strategy that works
| Model | Result | Verdict |
|---|---|---|
| Counterparty to every dump ≥ R$ 300k, ≥ 50 bps below fair, 21 d, hedged | +108 bps / trade, 76% hit | **The edge** |
| Same, entering the **next day** | −15 bps | The edge requires being the counterparty |
| Toxicity model (forced vs informed seller), walk-forward | p ≥ 0.8: 79% hit, +126 bps; passes a t−2 lag test | Kept (this notebook's model) |
| Stress-cycle feature (days since CDI credit funds started losing to CDI with redemptions) | First 20 bd after the start: +29 bps; 40-60 bd: +139 | Kept (`dse`) |
| Walk-forward hedge beta | IPCA+ debentures stopped following NTN-B in 2026 (beta 0.9 → 0.3); full hedge lost money | Kept |
| Late-day dumps (after 16:00) revert more, morning ones less | +21 bps (t 4.6) vs −16 (t −3.4) | Kept (`hour`) |
| 20 original ideas (issuer buyback, performance-fee pump, iceberg, quarter end, holiday eve, round rate, name twins, rain...) | Only late-day survived \|t\| > 3 | Dropped |
| 10 "Hail Mary" ideas | Repeated same-day dumping = informed (t −5.2); odd lots = informed (t −3.3) | Kept (`n_prior_today`, `round_lot`) |
| Every idea in one walk-forward model | Buy-every-dump ≈ the model; the model adds ≈ +0.5%/yr | The money is in being the counterparty |

### 2.4 Look-ahead traps we found (and the rule that now prevents each)
| Trap | What happened | Rule in this notebook |
|---|---|---|
| Kalman posterior | Used the print to judge the print | Fair value = information **before** the print (yesterday's mark) |
| Bid-ask bounce | Last print vs next print looked like 80-91% hit | Entry = the dump itself; exit = a later, different trade day |
| Mark catch-up | Same-day mark models 70-79% → 45-54% lagged | Every signal gets a 1-day-lag test (§ 10) |
| Spike cleaning using t+1 | Look-ahead | Only past data |
| "Lucky print" exits | Choosing a good exit print | Exit = first trade day after t + 21 bd, its VWAP |
| "Last dump of the day", same-day IBOV, "fund later liquidated" | Needs the rest of the day / the future | Removed |
| Hand-picking ideas after seeing results | In-sample Sharpe 3.2 | All ideas go in one walk-forward model |
| Fund data used before publication | CVM data appears days later | Informe diário usable 6 bd later; CDA from its delivery date |
| 20% capture is order-dependent | ±1%/yr depending on which dumps are drawn | Average 10-30 random draws |

## 3. Setup and configuration

**Folders**
- `data/raw/`: raw downloads, exactly as received, one file per day / bond / year. Delete a file to force a re-download.
- `data/notebook/`: tables this notebook derives. Cached because the CVM step is slow; delete them to rebuild.
- `notebook_out/`: charts, trade lists, the live model, the bid sheet.

**Switches**
- `DOWNLOAD`: if `False`, the notebook never touches the network and uses only what is in `data/raw`.
- `SOURCE`: the B3 tape and the ANBIMA marks can come from public collection (`"public"`) or from an internal SQL database (`"sql"`). With `"sql"`, the query helper `sql_func.try_retry(query)` must be importable, and every query is limited to one month. SQL is better for marks because it includes bonds that have since matured or defaulted; the public ANBIMA history only has bonds that are still alive.

Requirements: `pandas numpy pyarrow requests lightgbm scikit-learn matplotlib joblib`, plus `selenium` with Microsoft Edge for the ANBIMA history step.

In [ ]:
import io, os, re, sys, json, time, zipfile, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

# ----------------------------------------------------------------------------- folders
ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"                       # raw downloads (kept as received)
WORK = ROOT / "data" / "notebook"                 # tables derived by this notebook
OUT = ROOT / "notebook_out"                       # charts, trades, live model
for p in (RAW, WORK, OUT): p.mkdir(parents=True, exist_ok=True)
CVM_DIRS = [RAW / "cvm", Path(os.environ.get("LOCALAPPDATA", "")) / "cvm_cache"]   # where CVM zips may already be

# ----------------------------------------------------------------------------- switches
DOWNLOAD = True                                   # False = offline, cached raw files only
TODAY = pd.Timestamp.today().normalize()
SOURCE = {"trades": "public", "anbima": "public"} # "public" = scraped / downloaded here | "sql" = internal tables
SQL = {                                           # only used when SOURCE[...] == "sql"
    "func_dir": os.environ.get("SQL_FUNC_DIR", str(ROOT)),     # folder with sql_func.py exposing try_retry(query) -> DataFrame
    "trades_table": "B3_CREDITO_OPERACOES",
    "trades_cols": {"date": "datanegocio", "time": "horanegocio", "code": "codigoif", "type": "instrumentofinanceiro",
                    "qty": "qtdadenegociada", "pu": "preconegocio", "vol": "volumefinanceiro", "rate": "taxanegocio"},
    "trades_types": ("DEB",),
    "anbima_table": "tbhh3003_anbima_debenture_mercado_secundario",
    "anbima_cols": {"date": "data_referencia", "code": "codigo_ativo", "rate": "taxa_indicativa", "pu": "pu",
                    "pct_par": "percent_pu_par", "duration": "duration"},       # duration in business days
}
ANBIMA_MAX_FETCH = 400                            # max bonds whose ANBIMA history is (re)downloaded per run (slow: browser)

# ----------------------------------------------------------------------------- strategy parameters (the backtested defaults)
B3_START = pd.Timestamp("2023-03-13")             # first day of the public B3 trade-by-trade export
START, OOS_START = "2023-04-01", "2024-01-01"     # dataset start; model is out of sample from 2024
MIN_VOL, MIN_DISC, MAX_DISC = 3e5, 50, 2000       # dump = >= R$ 300k at 50-2000 bps (price) below fair
HOLD_BD, COST_BPS, LAG_BD = 21, 1.0, 1            # holding period; cost per side (bps); fair value from t - LAG_BD
CAPITAL, CAPTURE, CUT = 50e6, 0.20, 0.70          # fund size; share of dumps that reach the desk; buy if p >= CUT
CAP_BOND, CAP_ISSUER = 0.04, 0.15                 # max share of capital per bond / issuer
DISTRESS_PCT, CAP_DISTRESS = 90, 0.01             # bond or issuer's worst bond < 90% of par -> max 1% per bond
BLOCK_MIN, CAP_BLOCK = 3e6, None                  # scaling test: dumps >= BLOCK_MIN get CAP_BLOCK (None = CAP_BOND)
N_DRAWS = 10                                      # random capture draws per result
USE_FUND_STRESS = True                            # CVM credit-fund stress feature
RUN_FULL_ROBUSTNESS = True                        # False skips the slow rebuilds in section 10
LGB_PARAMS = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=200, subsample=0.8,
                  subsample_freq=1, colsample_bytree=0.7, reg_lambda=10.0, random_state=0, verbose=-1)
FEATS = ["disc", "log_vol", "size_adv", "dur", "fair", "is_cdi", "is_ipca", "is_pre", "days_since", "mom5", "mom20",
         "iss_mom5", "iss_min", "fail_bond90", "fail_iss90", "hour", "n_prior_today", "round_lot", "dse"]

# ----------------------------------------------------------------------------- small helpers
S = requests.Session(); S.headers["User-Agent"] = "Mozilla/5.0 (research notebook)"

def log(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)

def get(url, tries=4, timeout=300, **kw):
    """GET with retries; returns the response or None. Never called when DOWNLOAD is False."""
    if not DOWNLOAD: return None
    for k in range(tries):
        try:
            r = S.get(url, timeout=timeout, **kw)
            if r.status_code == 200: return r
        except requests.RequestException:
            pass
        time.sleep(2 * (k + 1))
    return None

def num_br(s):
    """Brazilian numbers '1.234,56' -> 1234.56 (also accepts '1234.56' and floats)."""
    s = pd.Series(s).astype(str).str.strip()
    br = s.str.contains(",", regex=False)
    return pd.to_numeric(s.where(~br, s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)), errors="coerce")

def cnpj(s):
    """CNPJ (company tax id) as 14 digits; the first 8 digits (the 'root') identify the company group."""
    return pd.Series(s).astype(str).str.replace(r"\D", "", regex=True).str.zfill(14)

def _sql(query):
    sys.path.insert(0, SQL["func_dir"])
    from sql_func import try_retry
    df = try_retry(query); df = df if isinstance(df, pd.DataFrame) else pd.DataFrame(df)
    df.columns = [str(c).lower() for c in df.columns]; return df

def _sql_by_month(build_query):
    """Every SQL query is limited to one month."""
    return pd.concat([_sql(build_query(m.start_time.strftime("%Y-%m-%d"), m.end_time.strftime("%Y-%m-%d")))
                      for m in pd.period_range(START, TODAY, freq="M")], ignore_index=True)
print("root:", ROOT, "| download:", DOWNLOAD, "| sources:", SOURCE)

## 4. Data collection

Each subsection collects one dataset. For each we state what it is, where it comes from, how it is downloaded, which columns we keep, and **when the information became public**. That last point is the point-in-time rule that prevents look-ahead.

| # | Dataset | Source (free, no login) | Used for | Known from |
|---|---|---|---|---|
| 4.1 | Bond characteristics | SND, debentures.com.br | Indexer, issuer, coupon, maturity, payment schedule | Static |
| 4.2 | B3 trade-by-trade tape | B3 BDI export, arquivos.b3.com.br | Dumps, entry and exit prices, size, time of day, liquidity | Same day (published every 15 min) |
| 4.3 | ANBIMA marks, history per bond | data.anbima.com.br public bond page | Fair value, momentum, price level | Evening of the same day → used from t+1 |
| 4.4 | ANBIMA daily file | anbima.com.br daily `db` file (last ~20 bd only) | Recent marks, ANBIMA's duration (to validate ours) | Evening of the same day |
| 4.5 | SND payment events | SND "PU de eventos" | Skip windows with cash events; payment failures (defaults) | Event date |
| 4.6 | Tesouro Direto rates | Tesouro Transparente CSV | Rates hedge (pre and IPCA+ curves) | Same day |
| 4.7 | CDI and IPCA | Central Bank SGS API (series 12, 433) | Cash return, benchmark, hedge carry | Same day / monthly |
| 4.8 | Fund NAV, flows, holdings | CVM open data (informe diário, CDA, delivery log, fund registry) | Credit-fund stress episodes | Informe +6 bd; CDA at its delivery date |

### 4.1 Bond characteristics (SND)

**What.** One row per debenture ever registered: code, issuer and CNPJ, ISIN, issue and maturity dates, indexer text (e.g. "DI", "IPCA"), multiplier (% of CDI) or spread, incentivized-bond flag (Law 12,431), and the amortization and coupon schedule fields.

**Where.** `debentures.com.br › Explore o SND › Consulta a dados › Características`. One tab-separated export for active bonds (`op_exc=False`) and one for bonds that left the system (`op_exc=True`: matured, redeemed, defaulted). We need both, otherwise dead bonds in the tape would have no indexer.

**How we classify the indexer:**
- **IPCA+**: the index text contains IPCA or IGP.
- **%CDI**: it contains DI, the multiplier is > 100% and there is no spread.
- **CDI+**: any other DI bond.
- **PRE**: the text says PRE or is empty.

The issuer **root** (first 8 digits of the CNPJ) groups different legal entities of the same company. It is used for the issuer limits.

In [ ]:
SND = "https://www.debentures.com.br/exploreosnd/consultaadados/"

def snd_characteristics():
    """Raw SND characteristics export (active + excluded bonds), cached as one parquet."""
    f = RAW / "snd_caracteristicas_full.parquet"
    if f.exists() and (not DOWNLOAD or time.time() - f.stat().st_mtime < 7 * 86400):   # refresh weekly
        return pd.read_parquet(f)
    parts = []
    for exc in ("False", "True"):
        r = get(SND + f"emissoesdedebentures/caracteristicas_e.asp?tip_deb=publicas&op_exc={exc}&ativo=")
        if r is None: return pd.read_parquet(f)
        t = r.content.decode("latin1").splitlines()
        i = next(j for j, l in enumerate(t) if l.startswith("Codigo do Ativo"))       # skip the text header
        a = pd.read_csv(io.StringIO("\n".join(t[i:])), sep="\t", dtype=str, quoting=3)
        a.columns = [c.strip() for c in a.columns]
        parts.append(a.apply(lambda s: s.str.strip()).assign(excluded=exc == "True"))
    a = pd.concat(parts, ignore_index=True); a.to_parquet(f); return a

SC = snd_characteristics()
dt = lambda c: pd.to_datetime(SC[c], format="%d/%m/%Y", errors="coerce")
BOND = pd.DataFrame({"code": SC["Codigo do Ativo"].str.upper(), "issuer": SC["Empresa"], "issuer_cnpj": cnpj(SC["CNPJ"]).values,
                     "excluded": SC.excluded, "issue_date": dt("Data de Emissao"), "maturity": dt("Data de Vencimento"),
                     "index_txt": SC["indice"], "pct_index": num_br(SC["Percentual Multiplicador/Rentabilidade"]).values,
                     "coupon": num_br(SC["Juros Criterio Novo - Taxa"]).values,
                     "incentivized": SC["Deb. Incent. (Lei 12.431)"].str.upper().str.startswith("S")})
BOND = BOND.sort_values("excluded").drop_duplicates("code", keep="first")        # active row wins if a code appears twice
ix = BOND.index_txt.fillna("").str.upper()
BOND["indexer"] = np.select([ix.str.contains("IPCA|IGP"), ix.str.contains("DI") & (BOND.pct_index.fillna(100) > 100.001) & (BOND.coupon.fillna(0) == 0),
                             ix.str.contains("DI"), ix.str.contains("PRE|PRÉ") | (ix == "")], ["IPCA+", "%CDI", "CDI+", "PRE"], "OTHER")
BOND["root"] = BOND.issuer_cnpj.str[:8]
BOND["spread"] = BOND.coupon                                                      # CDI+ spread (% per year) for the passive benchmark
BOND = BOND.set_index("code")
print(f"{len(BOND):,} debentures | by indexer: {BOND.indexer.value_counts().to_dict()} | incentivized: {int(BOND.incentivized.sum()):,}")
display(BOND[["issuer", "indexer", "coupon", "maturity", "root"]].head(5))

### 4.2 B3 trade-by-trade tape

**What.** Every secondary-market trade of private fixed income registered at B3: instrument type, code, quantity, PU, R$ volume, rate, origin (platform / voice / pre-registration), **time of the trade**, trade id and status. This is the most important dataset: it defines the dumps, our entry price, the size, the time of day and the exit price.

**Where.** B3's BDI "Negócio a negócio" table, exported as CSV with a POST to `arquivos.b3.com.br/bdi/table/export/csv`, one day per request. The export goes back to 13/03/2023, which is why the dataset starts in 2023. Before that date only daily aggregates exist, and those cannot identify a dump.

**Cleaning.**
- Keep debentures (`tipo == "DEB"`) only.
- Drop cancelled trades.
- Drop duplicate trade ids.
- Convert Brazilian number formats.

**Point in time.** The table is updated every 15 minutes during the day, so a trade is known at its own time. B3 may adjust it on D+1.

**SQL alternative.** `B3_CREDITO_OPERACOES`, with the column map in § 3. Check that it has a trade-time column; without one the time-of-day features drop out.

In [ ]:
B3_COLS = ["tipo", "emissor", "cod", "qtd", "pu", "vol", "taxa", "origem", "hora", "data", "id", "isn", "liq", "situacao"]

def b3_day(d):
    """Raw BDI trade file for day d -> data/raw/b3_trades/YYYYMMDD.parquet (all instruments, as received)."""
    g = RAW / "b3_trades" / f"{d:%Y%m%d}.parquet"
    if g.exists() or not DOWNLOAD: return g
    g.parent.mkdir(parents=True, exist_ok=True)
    try:
        r = S.post("https://arquivos.b3.com.br/bdi/table/export/csv?lang=pt-BR", timeout=300,
                   json={"Name": "Trade", "Date": f"{d:%Y-%m-%d}", "FinalDate": f"{d:%Y-%m-%d}", "ClientId": "", "Filters": {}})
        t = r.content.decode("latin1").splitlines()
        i = next((k for k, l in enumerate(t) if l.count(";") > 5), None)            # first line of the table (after the text)
        x = pd.read_csv(io.StringIO("\n".join(t[i:])), sep=";", dtype=str) if i is not None else pd.DataFrame()
        if len(x): x.columns = B3_COLS[:x.shape[1]]
        x.to_parquet(g)                                                             # holidays are saved empty (not retried)
    except Exception as e:
        log("b3 fail", d.date(), type(e).__name__)
    return g

def load_trades():
    """Debenture trades: date, time, code, qty, pu, vol (R$), rate."""
    if SOURCE["trades"] == "sql":
        c = SQL["trades_cols"]; cols = [v for v in c.values() if v]; types = ",".join(f"'{t}'" for t in SQL["trades_types"])
        T = _sql_by_month(lambda a, b: f"SELECT {', '.join(cols)} FROM {SQL['trades_table']} WHERE {c['date']} >= '{a}' "
                                       f"AND {c['date']} <= '{b}' AND {c['type']} IN ({types})")
        T = T.rename(columns={v.lower(): k for k, v in c.items() if v})
        if "time" not in T: T["time"] = None
    else:
        days = pd.bdate_range(B3_START, TODAY - pd.Timedelta(days=1))
        with ThreadPoolExecutor(4) as ex: files = list(ex.map(b3_day, days))
        P = []
        for g in files:
            if not g.exists(): continue
            x = pd.read_parquet(g)
            if not len(x) or "tipo" not in x: continue
            x = x[x.tipo.eq("DEB") & ~x.situacao.fillna("").str.contains("Cancel")]
            P.append(pd.DataFrame({"date": pd.to_datetime(x.data, dayfirst=True).values, "time": x.hora.values, "code": x.cod.str.upper().values,
                                   "qty": num_br(x.qtd).values, "pu": num_br(x.pu).values, "vol": num_br(x.vol).values,
                                   "rate": num_br(x.taxa).values, "origin": x.origem.values, "trade_id": x["id"].values}))
        T = pd.concat(P, ignore_index=True).drop_duplicates("trade_id")
    T["date"] = pd.to_datetime(T.date).dt.normalize(); T["code"] = T.code.astype(str).str.strip().str.upper()
    for k in ("qty", "pu", "vol", "rate"): T[k] = pd.to_numeric(T[k], errors="coerce")
    T = T[(T.pu > 0) & (T.vol > 0) & (T.date >= START)].drop_duplicates(["date", "time", "code", "qty", "pu"])
    return T[["date", "time", "code", "qty", "pu", "vol", "rate"]].sort_values(["date", "time"]).reset_index(drop=True)

t0 = time.time(); TAPE = load_trades()
print(f"{len(TAPE):,} debenture trades, {TAPE.date.min().date()} -> {TAPE.date.max().date()}, {TAPE.code.nunique():,} bonds, "
      f"time present {TAPE.time.notna().mean():.1%}, R$ {TAPE.vol.sum() / 1e9:,.0f} bn ({time.time() - t0:.0f}s)")
print("trades >= R$ 300k:", f"{(TAPE.vol >= 3e5).sum():,}", "| median trade R$", f"{TAPE.vol.median():,.0f}")
display(TAPE.tail(3))

### 4.3 and 4.4 ANBIMA marks

**What.** ANBIMA's daily indicative price (PU) for each debenture, plus its PU par. Fair value = PU indicative / PU par × 100 = **%PU par**.

**Where.**
1. **History per bond (4.3).** The public bond page `data.anbima.com.br/debentures/{code}/precos` draws a chart of the full price history. We open the page in a headless Edge browser, click "Total" and keep the JSON the page itself downloads for its chart (`grafico-pu`).
   - No API token and no direct API calls are used, only the page's own response.
   - One bond at a time, three browsers in parallel, cached per bond in `data/raw/anbima_hist/{code}.parquet`.
2. **Daily file (4.4).** ANBIMA publishes `db{yymmdd}.txt` each evening with every priced debenture: rate, PU, %PU par, duration. Only the last ~20 business days are online, so we archive each one we download. It extends the history to the latest days and gives ANBIMA's own duration, which we use to check our duration calculation in § 5.

**Survivorship warning.** The public history exists only for bonds ANBIMA still prices today. Dumps in bonds that later matured or defaulted have no fair value, so they drop out. That makes the public backtest **slightly optimistic in defaults and slightly pessimistic in sample size**. The SQL table `tbhh3003_anbima_debenture_mercado_secundario` has every bond and fixes this.

**Which bonds to fetch.** Bonds in the latest daily file (priced today) whose cached history is missing, or older than the oldest daily file we hold, so the daily files cannot fill the gap. At most `ANBIMA_MAX_FETCH` per run.

**Point in time.** A mark for day t is published on the evening of t. It is used only from t + 1 (`LAG_BD = 1`).

In [ ]:
def anbima_daily_file(d):
    """ANBIMA daily debenture file for day d (rate, PU, %PU par, duration of every priced bond). Kept forever once downloaded."""
    f = RAW / "anbima_daily" / f"db{d:%y%m%d}.txt"
    if not f.exists() and DOWNLOAD:
        r = get(f"https://www.anbima.com.br/informacoes/merc-sec-debentures/arqs/db{d:%y%m%d}.txt", tries=1, timeout=60)
        if r is not None and len(r.content) > 5000:
            f.parent.mkdir(parents=True, exist_ok=True); f.write_bytes(r.content)
    if not f.exists(): return None
    L = f.read_bytes().decode("latin1").splitlines()
    i = next(j for j, l in enumerate(L) if l.startswith("Código@") or l.startswith("C\xf3digo@"))
    a = pd.read_csv(io.StringIO("\n".join(L[i:])), sep="@", dtype=str)
    return pd.DataFrame({"date": d, "code": a.iloc[:, 0].str.strip().str.upper(), "rate": num_br(a.iloc[:, 6]).values,
                         "pu": num_br(a.iloc[:, 10]).values, "pct_par": num_br(a.iloc[:, 11]).values, "duration_du": num_br(a.iloc[:, 12]).values})

ANB_DAILY = pd.concat([x for x in map(anbima_daily_file, pd.bdate_range(TODAY - pd.Timedelta(days=45), TODAY)) if x is not None], ignore_index=True)
print(f"ANBIMA daily files: {ANB_DAILY.date.nunique()} days ({ANB_DAILY.date.min().date()} -> {ANB_DAILY.date.max().date()}), "
      f"{ANB_DAILY.code.nunique():,} bonds priced on the last day")

In [ ]:
HOOK = r"""
window.__cap=[];
const of=window.fetch; window.fetch=async(...a)=>{const r=await of(...a);try{const u=(a[0].url||a[0])+"";if(/grafico-pu/.test(u)){r.clone().text().then(t=>window.__cap.push({u,t}))}}catch(e){}return r};
const ox=XMLHttpRequest.prototype.open,os=XMLHttpRequest.prototype.send;
XMLHttpRequest.prototype.open=function(m,u){this.__u=u;return ox.apply(this,arguments)};
XMLHttpRequest.prototype.send=function(){this.addEventListener("load",()=>{if(/grafico-pu/.test(this.__u))window.__cap.push({u:this.__u,t:this.responseText})});return os.apply(this,arguments)};
"""
HIST = RAW / "anbima_hist"; HIST.mkdir(parents=True, exist_ok=True)

def _browser():
    from selenium import webdriver
    o = webdriver.EdgeOptions()
    for a in ("--headless=new", "--window-size=1300,900", "--disable-gpu"): o.add_argument(a)
    return webdriver.Edge(options=o)

def anbima_history_one(d, c):
    """Open the public price page of bond c, click 'Total', keep the chart JSON the page downloads. -> status string"""
    d.get(f"https://data.anbima.com.br/debentures/{c}/precos")
    for _ in range(30):
        time.sleep(0.5)
        if d.execute_script("return [...document.querySelectorAll('li')].some(e=>e.innerText.trim()=='Total')"): break
    else:
        return "no_chart"
    d.execute_script(HOOK)
    d.execute_script("[...document.querySelectorAll('li')].find(e=>e.innerText.trim()=='Total')?.click()")
    for _ in range(30):
        time.sleep(0.5)
        cap = d.execute_script("return window.__cap.length ? window.__cap[window.__cap.length-1].t : null")
        if cap:
            x = pd.DataFrame([{"date": p["data"], "pu_par": p.get("valor_pu_historico"), "pu_ind": p.get("valor_pu_indicativo"),
                               "event": "; ".join(e.get("tipo", "") for e in p.get("eventos") or []),
                               "event_value": sum(float(e.get("valor") or 0) for e in p.get("eventos") or [])} for p in json.loads(cap).get("pus", [])])
            if not len(x): return "empty"
            x["date"] = pd.to_datetime(x.date)
            for k in ("pu_par", "pu_ind"): x[k] = pd.to_numeric(x[k], errors="coerce")
            x.to_parquet(HIST / f"{c}.parquet"); return "ok"
    return "no_capture"

def anbima_history_worker(codes):
    d, st = None, {}
    for n, c in enumerate(codes):
        try:
            if d is None or n % 15 == 0:                     # fresh browser every 15 bonds (memory)
                if d is not None: d.quit()
                d = _browser()
            st[c] = anbima_history_one(d, c)
        except Exception as e:
            st[c] = type(e).__name__
            try: d.quit()
            except Exception: pass
            d = None
    if d is not None: d.quit()
    return st

if SOURCE["anbima"] == "public":
    last_day = ANB_DAILY[ANB_DAILY.date == ANB_DAILY.date.max()]
    oldest_daily = ANB_DAILY.date.min()
    def stale(c):
        f = HIST / f"{c}.parquet"
        return (not f.exists()) or pd.read_parquet(f, columns=["date"]).date.max() < oldest_daily
    todo = [c for c in sorted(set(last_day.code)) if c in BOND.index and stale(c)][:ANBIMA_MAX_FETCH]
    log(f"ANBIMA history: {last_day.code.nunique():,} bonds priced today, {len(todo)} to (re)fetch")
    if todo and DOWNLOAD:
        st = {}
        with ThreadPoolExecutor(3) as ex:
            for r in ex.map(anbima_history_worker, [todo[i::3] for i in range(3)]): st.update(r)
        log("fetch status:", pd.Series(st).value_counts().to_dict())

### 4.5 SND payment events

**What.** Every scheduled payment of every debenture (coupon, amortization, early redemption, premium) with its **settlement status**: paid (`LIQUIDADO`), not paid (`NÃO LIQUIDADO`), in default (`INADIMPLENTE`), or late (`ATRASO`).

**Used for:**
1. A holding window that contains a cash event is skipped, because the PU jumps on payment days.
2. No buying a bond whose bond or issuer failed a payment in the last 90 days.

**Where.** SND › Eventos financeiros › PU de eventos, one tab-separated export per year (2010 →). Column names change between years, so we map them by keyword.

**Point in time.** A failure is known on the event date.

In [ ]:
def snd_events(start=2010):
    d = RAW / "alt" / "snd_eventos"; d.mkdir(parents=True, exist_ok=True); out = []
    for y in range(start, TODAY.year + 1):
        f = d / f"{y}.txt"
        if (not f.exists() or y >= TODAY.year) and DOWNLOAD:                          # the current year is refreshed
            r = get(SND + "eventosfinanceiros/pudeeventos_e.asp", params={"op_exc": "Nada", "dt_ini": f"01/01/{y}", "dt_fim": f"31/12/{y}"})
            if r is not None: f.write_bytes(r.content)
        if not f.exists(): continue
        t = f.read_bytes().decode("latin1").splitlines()
        i = next((j for j, l in enumerate(t) if l.lower().startswith("data")), None)
        if i is None: continue
        x = pd.read_csv(io.StringIO("\n".join(t[i:])), sep="\t", dtype=str, quoting=3, index_col=False)
        x.columns = [c.strip() for c in x.columns]
        ren = {c: k for c in x.columns for p, k in (("data", "Data"), ("ativo", "Ativo"), ("tipo", "Tipo"), ("liquida", "Liquidacao")) if c.lower().startswith(p)}
        out.append(x.rename(columns=ren)[[v for v in ("Data", "Ativo", "Tipo", "Liquidacao") if v in ren.values()]])
    E = pd.concat(out, ignore_index=True).apply(lambda s: s.str.strip() if s.dtype == object else s)
    E = pd.DataFrame({"date": pd.to_datetime(E.Data, format="%d/%m/%Y", errors="coerce"), "code": E.Ativo.str.upper(),
                      "event": E.Tipo, "status": E.Liquidacao.fillna("").str.upper()}).dropna(subset=["date"])
    # failure = not settled or in default; "ATRASO" (late but later paid) is not counted as a failure
    E["failed"] = E.status.str.contains(r"O LIQUIDADO|INADIMPL", regex=True) & ~E.status.str.fullmatch("LIQUIDADO") & ~E.status.str.contains("ATRAV")
    return E

EVENTS = snd_events()
print(f"{len(EVENTS):,} payment events, {int(EVENTS.failed.sum()):,} failures in {EVENTS[EVENTS.failed].code.nunique():,} bonds")
display(EVENTS[EVENTS.failed].tail(3))

### 4.6 and 4.7 Tesouro curve, CDI and IPCA

- **Tesouro Direto** (`PrecoTaxaTesouroDireto.csv`, Tesouro Transparente): daily rates of every Tesouro Direto bond.
  - **Tesouro Prefixado** (LTN) gives the pre-fixed curve; **Tesouro IPCA+** (NTN-B Principal) gives the real curve. We use the morning sell rate, which is always quoted.
  - The hedge for a bond of duration D is a short position in a Tesouro bond of the same duration, interpolated along the curve.
- **CDI** (BCB SGS series 12, % per day): cash return, the benchmark, and the CDI part of every trade's excess return.
- **IPCA** (BCB SGS 433, % per month): the inflation accrual of the IPCA+ hedge leg.

In [ ]:
def tesouro():
    f = RAW / "alt" / "tesouro" / "td.csv"
    if DOWNLOAD and (not f.exists() or time.time() - f.stat().st_mtime > 86400):
        r = get("https://www.tesourotransparente.gov.br/ckan/dataset/df56aa42-484a-4a59-8184-7676580c81e3/resource/"
                "796d2059-14e9-44e3-80c9-2d9e30b405c1/download/PrecoTaxaTesouroDireto.csv")
        if r is not None: f.parent.mkdir(parents=True, exist_ok=True); f.write_bytes(r.content)
    d = pd.read_csv(f, sep=";", decimal=",")
    d["date"] = pd.to_datetime(d["Data Base"], dayfirst=True)
    d["T"] = (pd.to_datetime(d["Data Vencimento"], dayfirst=True) - d.date).dt.days / 365.25     # years to maturity
    d["rate"] = d["Taxa Venda Manha"].where(d["Taxa Venda Manha"] > 0)
    d = d.dropna(subset=["rate"]); d = d[d.date >= pd.Timestamp(START) - pd.Timedelta(days=30)]
    # {"PRE" | "IPCA+": {date: (maturities in years, rates in %)}}
    return {k: {dt_: (x.sort_values("T")["T"].values, x.sort_values("T").rate.values) for dt_, x in d[d["Tipo Titulo"] == n].groupby("date")}
            for k, n in (("PRE", "Tesouro Prefixado"), ("IPCA+", "Tesouro IPCA+"))}

def sgs(code_, since="01/01/2021"):
    """Central Bank time series (free JSON API), cached as raw JSON."""
    f = RAW / "alt" / f"sgs_{code_}_nb.json"
    if DOWNLOAD and (not f.exists() or time.time() - f.stat().st_mtime > 86400):
        r = get(f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{code_}/dados?formato=json&dataInicial={since}", timeout=120)
        if r is not None: f.write_bytes(r.content)
    s = pd.DataFrame(json.loads(f.read_bytes()))
    return pd.Series(s.valor.astype(float).values, index=pd.to_datetime(s.data, dayfirst=True))

CURVES = tesouro()
cdi_d = sgs(12) / 100                                                               # CDI per business day (decimal)
CDI = (1 + cdi_d).cumprod(); CDI = CDI.reindex(pd.bdate_range(CDI.index.min(), CDI.index.max())).ffill()   # cumulative CDI index
CDI_ANNUAL = (1 + cdi_d) ** 252 - 1
ip = sgs(433); IPCA = pd.Series(ip.values / 100, index=ip.index.to_period("M"))    # monthly inflation (decimal)
print(f"Tesouro curves: {len(CURVES['PRE'])} days pre, {len(CURVES['IPCA+'])} days IPCA+ | CDI to {CDI.index.max().date()} "
      f"(last {CDI_ANNUAL.iloc[-1]:.2%}/yr) | IPCA to {IPCA.index.max()}")

### 4.8 CVM fund data: NAV, flows and holdings

**Why.** A **credit-fund stress episode** starts when CDI credit funds lose to CDI over 21 bd and see net redemptions over 21 bd.
- Brazilian credit funds typically pay redemptions D+30, so the forced selling arrives 20-30 bd after the shock.
- Dumps in the first ~20 bd of an episode keep falling. Dumps 40-90 bd in revert the most.
- The feature `dse` (business days since the episode started) was the strongest addition to the base model.

**Four CVM datasets** (`dados.cvm.gov.br`, monthly or yearly zips):

| Dataset | File | What we take |
|---|---|---|
| Fund registry | `FI/CAD/DADOS/registro_fundo_classe.zip` | Map from old fund CNPJ to the class CNPJ. Funds re-registered under RCVM 175 in 2024-25, so one fund can appear under two ids. |
| CDA (monthly holdings) | `FI/DOC/CDA/DADOS/cda_fi_YYYYMM.zip` (before 2023: `HIST/cda_fi_YYYY.zip`) | Block 4 (`BLC_4`): debentures each fund holds, at market value. `PL` file: the fund's NAV. |
| Delivery log | `FI/DOC/ENTREGA/DADOS/...` | The date each fund actually delivered each month's CDA, which is when that holding became public. |
| Informe diário | `FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_YYYYMM.zip` | Daily NAV, quota, subscriptions and redemptions of every fund |

**Steps:**
1. **Credit fund** = debentures ≥ 20% of NAV in the latest CDA *already published* on that day.
2. Exclude **incentivized funds** (≥ 50% of their debentures under Law 12,431). They are IPCA+ funds that move with real rates, not credit.
3. Aggregate across those funds: the NAV-weighted daily quota return minus CDI, summed over 21 bd (`xr21`), and 21 bd net flow / NAV (`fl21`).
4. Daily quota returns are clipped at ±2% (data errors). Funds below R$ 1 mi are dropped.

**Point in time.**
- Informe diário: the data for day t is used from **t + 6 bd**. The measured delivery is median D+3, p90 D+6 calendar days.
- CDA: used from its first delivery. Where no delivery is logged, from month end + 90 days, the legal maximum.

The parsed result is cached in `data/notebook/` because reading ~5 years of daily fund files takes several minutes.

In [ ]:
CVM = "https://dados.cvm.gov.br/dados/"

def cvm_file(rel):
    """A CVM zip from a local cache if present, else downloaded to data/raw/cvm."""
    name = rel.split("/")[-1]
    for p in [d / name for d in CVM_DIRS]:
        if p.exists() and p.stat().st_size > 1000: return p
    r = get(CVM + rel, timeout=900)
    if r is None: log("missing", rel); return None
    (RAW / "cvm").mkdir(parents=True, exist_ok=True); (RAW / "cvm" / name).write_bytes(r.content); return RAW / "cvm" / name

def fund_map():
    z = zipfile.ZipFile(cvm_file("FI/CAD/DADOS/registro_fundo_classe.zip"))
    f = pd.read_csv(z.open("registro_fundo.csv"), sep=";", encoding="latin1", dtype=str)
    c = pd.read_csv(z.open("registro_classe.csv"), sep=";", encoding="latin1", dtype=str)
    m = c.merge(f[["ID_Registro_Fundo", "CNPJ_Fundo"]], on="ID_Registro_Fundo")
    m["old"], m["fund"] = cnpj(m.CNPJ_Fundo).values, cnpj(m.CNPJ_Classe).values
    m = m[(m.groupby("old").fund.transform("nunique") == 1) & (m.old != m.fund)].drop_duplicates("old")   # unambiguous re-registrations
    return dict(zip(m.old, m.fund))

def cda(fmap, first="2021-07"):
    """Debenture holdings (fund, ref_date, code, value) and NAV (fund, ref_date, pl) from monthly CDA files."""
    rels = [f"FI/DOC/CDA/DADOS/HIST/cda_fi_{y}.zip" for y in range(pd.Timestamp(first).year, 2023)]
    rels += [f"FI/DOC/CDA/DADOS/cda_fi_{m.strftime('%Y%m')}.zip" for m in pd.period_range("2023-01", TODAY.to_period("M") - 1, freq="M")]
    H, PL = [], []
    for rel in rels:
        p = cvm_file(rel)
        if p is None: continue
        z = zipfile.ZipFile(p)
        for n in [x for x in z.namelist() if "BLC_4" in x]:
            d = pd.read_csv(z.open(n), sep=";", encoding="latin1", dtype=str).rename(columns={"CNPJ_FUNDO": "CNPJ_FUNDO_CLASSE"})
            d = d[d.TP_APLIC.str.startswith("Deb", na=False) & d.CD_ATIVO.notna()]
            H.append(pd.DataFrame({"fund": cnpj(d.CNPJ_FUNDO_CLASSE).values, "ref_date": pd.to_datetime(d.DT_COMPTC).values,
                                   "code": d.CD_ATIVO.str.strip().str.upper().values, "value": pd.to_numeric(d.VL_MERC_POS_FINAL, errors="coerce").values}))
        for n in [x for x in z.namelist() if "_PL_" in x]:
            d = pd.read_csv(z.open(n), sep=";", encoding="latin1", dtype=str).rename(columns={"CNPJ_FUNDO": "CNPJ_FUNDO_CLASSE"})
            PL.append(pd.DataFrame({"fund": cnpj(d.CNPJ_FUNDO_CLASSE).values, "ref_date": pd.to_datetime(d.DT_COMPTC).values,
                                    "pl": pd.to_numeric(d.VL_PATRIM_LIQ, errors="coerce").values}))
    H, PL = pd.concat(H, ignore_index=True), pd.concat(PL, ignore_index=True)
    for X in (H, PL): X["fund"] = X.fund.map(lambda c: fmap.get(c, c))
    H = H[H.ref_date >= first].groupby(["fund", "ref_date", "code"], as_index=False).value.sum()
    return H, PL[PL.ref_date >= first].drop_duplicates(["fund", "ref_date"], keep="last")

def cda_delivery(fmap):
    """First delivery date of each fund's CDA per month (CVM delivery log, 2021+)."""
    rels = [f"FI/DOC/ENTREGA/DADOS/HIST/fi_entrega_documento_{y}.zip" for y in range(2021, 2025)]
    rels += [f"FI/DOC/ENTREGA/DADOS/fi_entrega_documento_{m.strftime('%Y%m')}.zip" for m in pd.period_range("2025-01", TODAY.to_period("M"), freq="M")]
    parts = []
    for rel in rels:
        p = cvm_file(rel)
        if p is None: continue
        z = zipfile.ZipFile(p)
        for n in [x for x in z.namelist() if "diario" not in x]:
            d = pd.read_csv(z.open(n), sep=";", encoding="latin1", dtype=str,
                            usecols=["CNPJ_Fundo_Classe", "Tipo_Documento", "Data_Fim_Competencia", "Data_Hora_Entrega"])
            d = d[d.Tipo_Documento.eq("CDA")]
            parts.append(pd.DataFrame({"fund": cnpj(d.CNPJ_Fundo_Classe).values, "ref_date": pd.to_datetime(d.Data_Fim_Competencia).values,
                                       "delivered": pd.to_datetime(d.Data_Hora_Entrega.str[:10]).values}))
    E = pd.concat(parts, ignore_index=True); E["fund"] = E.fund.map(lambda c: fmap.get(c, c))
    E["ref_date"] = E.ref_date + pd.offsets.MonthEnd(0)
    return E.groupby(["fund", "ref_date"]).delivered.min().rename("first_delivery").reset_index()

def informe(funds, fmap, first="2021-12"):
    """Daily NAV, quota, subscriptions, redemptions for the given funds (class level; subclasses summed when no class row)."""
    rev = {v: k for k, v in fmap.items()}; want = set(funds) | {rev[f] for f in funds if f in rev}
    F = []
    for m in pd.period_range(first, TODAY.to_period("M"), freq="M"):
        p = cvm_file(f"FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_{m.strftime('%Y%m')}.zip")
        if p is None: continue
        z = zipfile.ZipFile(p)
        for n in z.namelist():
            d = pd.read_csv(z.open(n), sep=";", encoding="latin1", dtype=str).rename(columns={"CNPJ_FUNDO": "CNPJ_FUNDO_CLASSE"})
            d = d[cnpj(d.CNPJ_FUNDO_CLASSE).isin(want).values]
            sub = d.ID_SUBCLASSE.fillna("").astype(str).str.strip().ne("").values if "ID_SUBCLASSE" in d else np.zeros(len(d), bool)
            F.append(pd.DataFrame({"fund": cnpj(d.CNPJ_FUNDO_CLASSE).values, "date": pd.to_datetime(d.DT_COMPTC).values, "sub": sub,
                                   "pl": pd.to_numeric(d.VL_PATRIM_LIQ, errors="coerce").values, "quota": pd.to_numeric(d.VL_QUOTA, errors="coerce").values,
                                   "capt": pd.to_numeric(d.CAPTC_DIA, errors="coerce").values, "resg": pd.to_numeric(d.RESG_DIA, errors="coerce").values}))
    F = pd.concat(F, ignore_index=True); F["fund"] = F.fund.map(lambda c: fmap.get(c, c))
    cls = F[~F["sub"]].set_index(["fund", "date"]).index
    Sb = F[F["sub"]].groupby(["fund", "date"], as_index=False).agg(pl=("pl", "sum"), capt=("capt", "sum"), resg=("resg", "sum"))
    Sb = Sb[~Sb.set_index(["fund", "date"]).index.isin(cls)].assign(quota=np.nan)          # subclass-only funds: no class quota
    F = pd.concat([F[~F["sub"]].drop(columns="sub"), Sb], ignore_index=True).drop_duplicates(["fund", "date"], keep="last")
    return F.sort_values(["fund", "date"])

f_cls, f_flw = WORK / "fund_class.parquet", WORK / "fund_daily.parquet"
if USE_FUND_STRESS and not (f_cls.exists() and f_flw.exists()):
    t0 = time.time(); fmap = fund_map(); log("fund map:", len(fmap), "re-registered funds")
    H, PL = cda(fmap); log("CDA:", f"{len(H):,} holdings rows,", H.fund.nunique(), "funds")
    Dlv = cda_delivery(fmap)
    inc = H.code.map(BOND.incentivized).fillna(False).astype(bool)
    Cl = H.assign(inc_value=H.value.where(inc, 0)).groupby(["fund", "ref_date"]).agg(deb_value=("value", "sum"), inc_value=("inc_value", "sum")).reset_index()
    Cl = Cl.merge(PL, on=["fund", "ref_date"], how="left").merge(Dlv, on=["fund", "ref_date"], how="left")
    Cl["pub_date"] = Cl.first_delivery.fillna(Cl.ref_date + pd.Timedelta(days=90))       # when this CDA became public
    Cl["credit_fund"] = Cl.deb_value / Cl.pl >= 0.2
    Cl["inc_fund"] = Cl.credit_fund & (Cl.inc_value / Cl.deb_value >= 0.5)
    Cl.to_parquet(f_cls)
    big = H.groupby("fund").value.max(); Fd = informe(set(big[big > 1e6].index), fmap); Fd.to_parquet(f_flw)
    log(f"informe diário: {len(Fd):,} rows, {Fd.fund.nunique():,} funds ({time.time() - t0:.0f}s)")
FUND_CLASS = pd.read_parquet(f_cls) if USE_FUND_STRESS else None
FUND_DAILY = pd.read_parquet(f_flw) if USE_FUND_STRESS else None
if USE_FUND_STRESS:
    last = FUND_CLASS[FUND_CLASS.ref_date == FUND_CLASS.ref_date.max()]
    print(f"latest CDA {last.ref_date.max().date()}: {int(last.credit_fund.sum())} credit funds, {int(last.inc_fund.sum())} of them incentivized | "
          f"informe diário {FUND_DAILY.date.min().date()} -> {FUND_DAILY.date.max().date()}")

## 5. Cleaning: the tables the model uses

### 5.1 Fair-value marks
`MARKS` has one row per (bond, day): `pct_par` (fair value), `pu_par` (to convert any trade's PU into %PU par) and `dur`.
- **History rows**: %PU par = PU indicative / PU par × 100, from the per-bond pages.
- **Daily-file rows** fill the days the history does not have: PU par = PU / (%PU par / 100).
- With `SOURCE["anbima"] = "sql"`, the internal table replaces both.

In [ ]:
def load_marks():
    if SOURCE["anbima"] == "sql":
        c = SQL["anbima_cols"]
        A = _sql_by_month(lambda a, b: f"SELECT {', '.join(c.values())} FROM {SQL['anbima_table']} WHERE {c['date']} >= '{a}' AND {c['date']} <= '{b}'")
        A = A.rename(columns={v.lower(): k for k, v in c.items()})
        A["pct_par"] = pd.to_numeric(A.pct_par, errors="coerce"); A["pu_par"] = pd.to_numeric(A.pu, errors="coerce") / (A.pct_par / 100)
        A["anbima_dur"] = pd.to_numeric(A.duration, errors="coerce") / 252
    else:
        rows = []
        for f in HIST.glob("*.parquet"):
            x = pd.read_parquet(f)
            if len(x): rows.append(x.assign(code=f.stem.upper()))
        H = pd.concat(rows, ignore_index=True); H["date"] = pd.to_datetime(H.date)
        H = H[(H.pu_par > 0) & (H.pu_ind > 0)]
        A = pd.DataFrame({"date": H.date, "code": H.code, "pct_par": H.pu_ind / H.pu_par * 100, "pu_par": H.pu_par, "src": "history"})
        Dd = ANB_DAILY[(ANB_DAILY.pct_par > 0) & (ANB_DAILY.pu > 0)]
        Dd = pd.DataFrame({"date": Dd.date, "code": Dd.code, "pct_par": Dd.pct_par, "pu_par": Dd.pu / (Dd.pct_par / 100), "src": "daily file"})
        A = pd.concat([A, Dd], ignore_index=True).drop_duplicates(["code", "date"], keep="first")     # history wins on overlap
    A["date"] = pd.to_datetime(A.date).dt.normalize(); A["code"] = A.code.astype(str).str.strip().str.upper()
    A = A[(A.pct_par > 1) & (A.pu_par > 0) & (A.date >= pd.Timestamp(START) - pd.Timedelta(days=60))]
    return A.drop_duplicates(["code", "date"]).sort_values(["code", "date"]).reset_index(drop=True)

MARKS = load_marks()
print(f"{len(MARKS):,} marks, {MARKS.code.nunique():,} bonds, {MARKS.date.min().date()} -> {MARKS.date.max().date()} |",
      MARKS.src.value_counts().to_dict() if "src" in MARKS else "")
tr = TAPE[TAPE.vol >= MIN_VOL]
cov = tr.merge(MARKS[["code", "date"]], on=["code", "date"], how="left", indicator=True)._merge.eq("both").mean()
print(f"share of trades >= R$ 300k with a same-day mark (needed to price the trade in %PU par): {cov:.1%}  <- the rest are bonds ANBIMA no longer prices")

### 5.2 Duration from each bond's payment schedule

The hedge needs each bond's **duration** on the trade date, and duration is also a model feature. ANBIMA publishes it only in the daily file, so only for recent days. We compute it ourselves from the SND schedule:

1. **Schedule.** Amortization dates run every k months from the first amortization date ("Amortizacao - Carencia / Cada / Unidade"). Coupon dates follow the same pattern with the "Juros Criterio Novo" fields. Amortizations are split equally across their dates, since SND has no per-date table for variable schedules. A bond with no amortization fields is a bullet.
2. **Flat yield y.**
   - CDI+: current CDI + spread.
   - %CDI: CDI × multiplier.
   - IPCA+ and PRE: the bond's own coupon rate (real rate for IPCA+).
   - Duration is not very sensitive to y, so this approximation is adequate.
3. **Duration.** Macaulay = Σ tᵢ·PVᵢ / Σ PVᵢ over the remaining flows. **Spread duration** = Macaulay / (1 + y).
4. **Grid.** Computed on every Friday for each bond (and its last mark), then carried forward to the other days.

**Check.** We compare our Macaulay duration with ANBIMA's published duration in the daily files.

In [ ]:
def _step(unit, k):
    u = str(unit).upper()
    return pd.DateOffset(months=k) if u.startswith("MES") else pd.DateOffset(years=k) if u.startswith("ANO") else pd.DateOffset(days=k) if u.startswith("DIA") else None

def _dates(first, unit, every, mat):
    """Payment dates: every `every` `unit`s from `first` until maturity (maturity always included)."""
    try: k = int(str(every).strip() or 0)
    except ValueError: k = 0
    st = _step(unit, k) if k > 0 else None
    if pd.isna(first) or st is None: return [mat]
    out, n = [], 0
    while first + st * n < mat - pd.Timedelta(days=5) and n < 400: out.append(first + st * n); n += 1
    return out + [mat]

def schedules(SC):
    dtc = lambda c: pd.to_datetime(SC[c], format="%d/%m/%Y", errors="coerce")
    A = SC.assign(code=SC["Codigo do Ativo"].str.upper(), mat=dtc("Data de Vencimento"), a_first=dtc("Amortizacao - Carencia"),
                  j_first=dtc("Juros Criterio Novo - Carencia")).sort_values("excluded").drop_duplicates("code")
    out = {}
    for r in A.to_dict("records"):
        if pd.isna(r["mat"]): continue
        am = _dates(r["a_first"], r.get("Amortizacao - Unidade"), r.get("Amortizacao - Cada"), r["mat"])
        jd = set(_dates(r["j_first"], r.get("Juros Criterio Novo - Unidade"), r.get("Juros Criterio Novo - Cada"), r["mat"]))
        ds = sorted(set(am) | jd)
        out[r["code"]] = (np.array(ds, dtype="datetime64[ns]"), np.array([1 / len(am) if d in am else 0.0 for d in ds]), np.array([d in jd for d in ds]))
    return out

def duration(sched, code_, t, y, macaulay=False):
    """Duration (years) of bond code_ at date t with flat yield y (decimal)."""
    if code_ not in sched or not np.isfinite(y): return np.nan
    d, a, cp = sched[code_]; m = d > np.datetime64(t)
    if not m.any(): return np.nan
    d, a, cp = d[m], a[m], cp[m]
    rem = a / a.sum()                                            # remaining notional normalised to 1
    notional = 1 - np.concatenate([[0], np.cumsum(rem)[:-1]])    # notional outstanding before each date
    tt = (d - np.datetime64(t)).astype("timedelta64[D]").astype(float) / 365.25
    acc, last_c = np.zeros(len(tt)), 0.0
    for j in range(len(tt)):                                     # coupon = interest accrued since the previous coupon date
        if cp[j]: acc[j] = notional[j] * ((1 + y) ** (tt[j] - last_c) - 1); last_c = tt[j]
    pv = (rem + acc) / (1 + y) ** tt
    mac = (tt * pv).sum() / pv.sum()
    return mac if macaulay else mac / (1 + y)

def flat_yield(codes, dates):
    ixr = codes.map(BOND.indexer); cpn = codes.map(BOND.coupon).fillna(0) / 100; mult = codes.map(BOND.pct_index).fillna(100) / 100
    cdi = CDI_ANNUAL.reindex(pd.DatetimeIndex(dates), method="ffill").values
    y = np.select([ixr.eq("CDI+"), ixr.eq("%CDI")], [cdi + cpn, cdi * mult], np.where(cpn > 0, cpn, np.where(ixr.eq("IPCA+"), 0.06, 0.12)))
    return np.clip(y, 0.0, 0.5)

SCHED = schedules(SC)
G = MARKS[(MARKS.date.dt.dayofweek == 4) | (MARKS.groupby("code").date.transform("max") == MARKS.date)][["code", "date"]].copy()
t0 = time.time()
G["dur"] = [duration(SCHED, c, t, y) for c, t, y in zip(G.code, G.date, flat_yield(G.code, G.date))]
MARKS = MARKS.merge(G, on=["code", "date"], how="left")
MARKS["dur"] = MARKS.groupby("code").dur.ffill()
print(f"durations: {G.dur.notna().sum():,} grid points in {time.time() - t0:.0f}s; median {MARKS.dur.median():.2f} years")
# check against ANBIMA's published (Macaulay, business-day) duration on the latest daily file
chk = ANB_DAILY[ANB_DAILY.date == ANB_DAILY.date.max()].dropna(subset=["duration_du"]).copy()
chk = chk[chk.code.isin(SCHED.keys()) & (chk.duration_du > 0)]
chk["ours"] = [duration(SCHED, c, t, y, macaulay=True) for c, t, y in zip(chk.code, chk.date, flat_yield(chk.code, chk.date))]
chk["anbima"] = chk.duration_du / 252; chk = chk.dropna(subset=["ours"])
print(f"vs ANBIMA ({len(chk)} bonds): median abs error {(chk.ours - chk.anbima).abs().median():.2f} y, correlation {chk.ours.corr(chk.anbima):.3f}")

### 5.3 Credit-fund stress episodes

From the CVM tables, build the daily aggregate of CDI credit funds:
- `xr21` = 21-bd sum of (NAV-weighted quota return − CDI);
- `fl21` = 21-bd net flow / NAV.

A day is **stressed** when both are negative. An **episode** starts on a stressed day with no stressed day in the previous 90 calendar days.

Every value is stamped with `known` = date + 6 bd, the day the data becomes public. The feature `dse` on a trade date is the number of business days since the latest episode start that was already known on that date (999 if more than 120).

In [ ]:
def stress_series():
    Q = FUND_DAILY[FUND_DAILY.date >= "2022-01-01"].sort_values("date")
    F = FUND_CLASS[["fund", "pub_date", "credit_fund", "inc_fund"]].dropna().sort_values("pub_date")
    F["credit_fund"] = F.credit_fund.astype(bool) & ~F.inc_fund.astype(bool)          # CDI credit funds only
    # each fund-day gets the classification of the latest CDA already PUBLIC on that day
    Q = pd.merge_asof(Q, F[["fund", "pub_date", "credit_fund"]].rename(columns={"pub_date": "date"}), on="date", by="fund", direction="backward")
    Q = Q[Q.credit_fund.fillna(False).astype(bool)].sort_values(["fund", "date"])
    Q["r"] = Q.groupby("fund").quota.pct_change(fill_method=None).clip(-0.02, 0.02)   # daily quota return, data errors clipped
    Q["pl0"] = Q.groupby("fund").pl.shift(1)
    Q = Q.dropna(subset=["r", "pl0"]); Q = Q[Q.pl0 > 1e6]
    Q["net"] = Q.capt.fillna(0) - Q.resg.fillna(0)
    A = Q.assign(rw=Q.r * Q.pl0).groupby("date").agg(rw=("rw", "sum"), net=("net", "sum"), pl=("pl0", "sum"), n_funds=("fund", "nunique"))
    A["xr"] = A.rw / A.pl - cdi_d.reindex(A.index).ffill()
    A["xr21"] = A.xr.rolling(21).sum(); A["fl21"] = A.net.rolling(21).sum() / A.pl
    A["known"] = A.index + pd.offsets.BDay(6)                                          # public 6 business days later
    return A

def episode_starts(A):
    starts, last = [], None
    for d, v in zip(A.known, (A.xr21 < 0) & (A.fl21 < 0)):
        if v and (last is None or (d - last).days > 90): starts.append(d)
        if v: last = d
    return pd.Series(pd.to_datetime(starts))

STRESS_AGG = stress_series() if USE_FUND_STRESS else None
STRESS = episode_starts(STRESS_AGG) if USE_FUND_STRESS else None
if USE_FUND_STRESS:
    print(f"credit-fund aggregate: {STRESS_AGG.n_funds.median():.0f} funds/day, NAV R$ {STRESS_AGG.pl.iloc[-1] / 1e9:,.0f} bn | episode starts (known dates):",
          [str(d.date()) for d in STRESS])
    fig, ax = plt.subplots(figsize=(11, 3.4), dpi=110)
    ax.plot(STRESS_AGG.known, STRESS_AGG.xr21 * 1e4, lw=1.2, color="#2a78d6", label="xr21: credit funds vs CDI, 21 bd (bps)")
    ax.plot(STRESS_AGG.known, STRESS_AGG.fl21 * 1e4 / 10, lw=1.2, color="#d97a29", label="fl21: net flow / NAV, 21 bd (bps / 10)")
    for d in STRESS: ax.axvline(d, color="#e34948", lw=1, ls="--")
    ax.axhline(0, color="k", lw=0.6); ax.legend(frameon=False, fontsize=8); ax.spines[["top", "right"]].set_visible(False)
    ax.set_title("Credit-fund stress: red lines = episode starts (date the data became public)", loc="left", fontsize=9)
    plt.tight_layout(); plt.show()

### 5.4 Everything in one place

From here on the notebook uses only `DATA`:
- `T`: tape;
- `A`: marks with duration;
- `B`: bonds;
- `E`: events;
- `C`: Tesouro curves;
- `ipca`;
- `stress`: episode starts.

In [ ]:
DATA = dict(T=TAPE, A=MARKS[["date", "code", "pct_par", "pu_par", "dur"]], B=BOND[["indexer", "root", "spread"]],
            E=EVENTS[["code", "date", "failed"]], C=CURVES, ipca=IPCA, stress=STRESS)
print({k: (len(v) if hasattr(v, "__len__") else v) for k, v in DATA.items()})

## 6. Building the dataset: dumps, outcomes, hedge, features

`build()` turns the tables into one row per **dump**, with its outcome and its features.

**1. Find the dumps.**
- Every trade ≥ `min_vol` is priced in %PU par with the same day's PU par. The accrual schedule is contractual, so this is not look-ahead.
- `fair` = the bond's mark from t − `LAG_BD` (yesterday).
- `disc` = (fair − trade %PU par) / fair × 10,000, in bps. A dump has `disc` between `min_disc` and `max_disc`.
- We build with a loose filter (≥ R$ 100k, 30-5000 bps) so § 10 can vary the definition without rebuilding. `dumps()` applies the configured filter.

**2. Exit.**
- The first day the bond trades after t + `hold_bd` (searched up to 14 calendar days further), at that day's VWAP.
- If no trade comes, there is no exit and the dump is dropped. A live fund would still hold it; § 10 shows this barely matters.
- If an SND cash event falls between entry and exit, the dump is dropped, because the PU jumps on payment days.

**3. Return over CDI.** `ex` = PU_exit / PU_entry − CDI_exit / CDI_entry − 2 × cost.

**4. Rates hedge (IPCA+, PRE).**
- `tes` = excess return over CDI of holding a Tesouro bond of the same duration: carry (1 + y₀)^(bd/252), inflation for IPCA+, and the price change −D·(y₁ − y₀), all minus CDI.
- The hedge ratio `beta` per indexer and quarter = cov(ex, tes) / var(tes), estimated on trades **closed** in the previous 12 months (at least 300), clipped to [0, 1.2].
- Hedged return `ret` = ex − beta × tes. Label `y` = 1 if ret > 0.

**5. Features**, all known at the print:

| Feature | Meaning |
|---|---|
| `disc` | Discount to yesterday's fair value, bps |
| `log_vol`, `size_adv` | Trade size in R$ (log) and as a multiple of the bond's 63-bd ADV up to yesterday |
| `dur`, `fair` | Duration; yesterday's %PU par (price level: distressed bonds sit far below 100) |
| `is_cdi`, `is_ipca`, `is_pre` | Indexer |
| `days_since` | Calendar days since the bond last traded (before today) |
| `mom5`, `mom20` | Change of the bond's mark over 5 and 20 bd up to yesterday |
| `iss_mom5`, `iss_min` | Median 5-bd mark change and the lowest %PU par among the issuer's bonds |
| `fail_bond90`, `fail_iss90` | SND payment failure of the bond or issuer in the previous 90 days |
| `hour` | Time of the trade (late-day dumps revert more) |
| `n_prior_today` | Cheap prints of this bond earlier today (repeated dumping = informed seller) |
| `round_lot` | Quantity multiple of 10 (odd lots were more informed) |
| `dse` | Business days since the latest known credit-fund stress episode start (999 = none in 120 bd) |

In [ ]:
def curve_at(C, kind, dates, durs):
    """Tesouro rate (decimal) at maturity = duration, on the latest curve date <= each date."""
    keys = np.array(sorted(C[kind]), dtype="datetime64[ns]")
    idx = np.searchsorted(keys, np.asarray(dates, dtype="datetime64[ns]"), side="right") - 1
    out = np.full(len(idx), np.nan)
    for i, (j, d) in enumerate(zip(idx, durs)):
        if j >= 0 and np.isfinite(d):
            Tm, r = C[kind][pd.Timestamp(keys[j])]; out[i] = np.interp(d, Tm, r) / 100
    return out

def build(hold_bd=HOLD_BD, lag_bd=LAG_BD, min_vol=1e5, min_disc=30, max_disc=5000, cost_bps=COST_BPS):
    T, A, B, E = DATA["T"].copy(), DATA["A"].copy(), DATA["B"], DATA["E"]
    C, ipca, stress = DATA["C"], DATA["ipca"], DATA["stress"]
    T["indexer"], T["root"] = T.code.map(B.indexer), T.code.map(B.root)
    T = T[T.indexer.isin(["CDI+", "%CDI", "IPCA+", "PRE"])]
    Dly = T.assign(pv=T.pu * T.vol).groupby(["code", "date"]).agg(pv=("pv", "sum"), vol=("vol", "sum")).reset_index()
    Dly["pu_vwap"] = Dly.pv / Dly.vol                                            # daily VWAP (used for exits)
    # 1. dumps: print %PU par vs yesterday's mark
    P = T[T.vol >= min_vol].merge(A[["code", "date", "pu_par"]], on=["code", "date"], how="left")
    P["pct"] = P.pu / P.pu_par * 100
    P["tm1"] = P.date - pd.offsets.BDay(lag_bd)
    A1 = A.rename(columns={"date": "tm1", "pct_par": "fair"})[["code", "tm1", "fair", "dur"]]
    P = pd.merge_asof(P.sort_values("tm1"), A1.sort_values("tm1"), on="tm1", by="code", direction="backward", tolerance=pd.Timedelta(days=7))
    P["disc"] = (P.fair - P.pct) / P.fair * 1e4
    P = P[P.disc.between(min_disc, max_disc)].copy()
    # 2. exit: daily VWAP of the first trade day after t + hold_bd; no cash event in between
    days = {k: (g.date.values, g.pu_vwap.values) for k, g in Dly.groupby("code")}
    evs = {k: np.sort(g.date.values) for k, g in E.groupby("code")}
    te, pu_out = [], []
    for code_, t in zip(P.code.values, P.date.values):
        d, pu = days.get(code_, (np.array([], "datetime64[ns]"), None))
        tgt = np.datetime64(pd.Timestamp(t) + pd.offsets.BDay(hold_bd))
        j = np.searchsorted(d, tgt, side="right")
        ok = j < len(d) and d[j] <= tgt + np.timedelta64(14, "D")
        if ok and code_ in evs:
            e = evs[code_]; k = np.searchsorted(e, t, side="right"); ok = not (k < len(e) and e[k] <= d[j])
        te.append(d[j] if ok else np.datetime64("NaT")); pu_out.append(pu[j] if ok else np.nan)
    P["te"], P["pu_out"] = pd.to_datetime(te), pu_out
    P = P.dropna(subset=["te"]).copy()
    # 3. excess return over CDI after cost
    P["ex"] = P.pu_out / P.pu - P.te.map(CDI).values / P.date.map(CDI).values - 2 * cost_bps / 1e4
    # 4. rates hedge leg (IPCA+ / PRE): Tesouro excess over CDI (carry + inflation + price)
    P["tes"] = 0.0
    for k in ("PRE", "IPCA+"):
        m = (P.indexer == k) & P.dur.notna()
        y0 = curve_at(C, k, P.loc[m, "date"].values, P.loc[m, "dur"].values)
        y1 = curve_at(C, k, P.loc[m, "te"].values, P.loc[m, "dur"].values)
        bd = np.busday_count(P.loc[m, "date"].values.astype("datetime64[D]"), P.loc[m, "te"].values.astype("datetime64[D]"))
        infl = np.ones(m.sum())
        if k == "IPCA+":   # IPCA of the entry month, pro rata (21 bd ~ 1 month)
            infl = np.array([(1 + ipca.get(a, ipca.iloc[-1])) ** (b_ / 21) for a, b_ in zip(P.loc[m, "date"].dt.to_period("M"), bd)])
        cdi_acc = P.loc[m, "te"].map(CDI).values / P.loc[m, "date"].map(CDI).values
        P.loc[m, "tes"] = (1 + y0) ** (bd / 252) * infl - cdi_acc - P.loc[m, "dur"].values * (y1 - y0)
    P = P.dropna(subset=["tes"])
    P["beta"] = 1.0                                     # walk-forward hedge ratio from trades closed in the prior 12 months
    for k in ("PRE", "IPCA+"):
        for q in pd.period_range(P.date.min(), P.date.max(), freq="Q"):
            past = P[(P.indexer == k) & (P.te < q.start_time) & (P.te >= q.start_time - pd.Timedelta(days=365))]
            if len(past) >= 300 and past.tes.var() > 0:
                P.loc[(P.indexer == k) & (P.date.dt.to_period("Q") == q), "beta"] = float(np.clip(np.cov(past.ex, past.tes)[0, 1] / past.tes.var(), 0, 1.2))
    P["ret"] = P.ex - P.beta * P.tes
    P["y"] = (P.ret > 0).astype(float)
    # 5. features (all known at the print)
    A = A.sort_values(["code", "date"])
    A["mom5"] = A.pct_par - A.groupby("code").pct_par.shift(5); A["mom20"] = A.pct_par - A.groupby("code").pct_par.shift(20)
    A["root"] = A.code.map(B.root)
    iss = A.groupby(["root", "date"]).agg(iss_mom5=("mom5", "median"), iss_min=("pct_par", "min")).reset_index().rename(columns={"date": "tm1"})
    P = pd.merge_asof(P.sort_values("tm1"), A[["code", "date", "mom5", "mom20"]].rename(columns={"date": "tm1"}).sort_values("tm1"),
                      on="tm1", by="code", direction="backward", tolerance=pd.Timedelta(days=7))
    P = pd.merge_asof(P.sort_values("tm1"), iss.sort_values("tm1"), on="tm1", by="root", direction="backward", tolerance=pd.Timedelta(days=7))
    Dly = Dly.sort_values(["code", "date"])
    Dly["adv63"] = Dly.groupby("code").vol.transform(lambda s: s.rolling(63, min_periods=1).sum()) / 63
    Dly["last_trade"] = Dly.date
    P = pd.merge_asof(P.sort_values("tm1"), Dly[["code", "date", "adv63", "last_trade"]].rename(columns={"date": "tm1"}).sort_values("tm1"),
                      on="tm1", by="code", direction="backward")
    P["days_since"] = (P.date - P.last_trade).dt.days
    P["size_adv"] = P.vol / P.adv63; P["log_vol"] = np.log(P.vol)
    for k, nm in (("CDI+", "is_cdi"), ("IPCA+", "is_ipca"), ("PRE", "is_pre")): P[nm] = (P.indexer == k).astype(float)
    tm = P.time.astype(str)
    P["hour"] = pd.to_numeric(tm.str[:2], errors="coerce") + pd.to_numeric(tm.str[3:5], errors="coerce") / 60
    P = P.sort_values(["code", "date", "time"])
    P["n_prior_today"] = P.groupby(["code", "date"]).cumcount()
    P["round_lot"] = (P.qty % 10 == 0).astype(float)
    F = E[E.failed].copy(); F["root"] = F.code.map(B.root)
    for key, nm in (("code", "fail_bond90"), ("root", "fail_iss90")):
        ev = F.groupby(key).date.apply(lambda s: np.sort(s.values)).to_dict()
        P[nm] = [float((e := ev.get(k)) is not None and bool(((e < np.datetime64(t)) & (e >= np.datetime64(t - pd.Timedelta(days=90)))).any()))
                 for k, t in zip(P[key], P.date)]
    if stress is not None:
        s = pd.DataFrame({"date": stress, "ep": stress}).sort_values("date")
        P = pd.merge_asof(P.sort_values("date"), s, on="date", direction="backward")
        dse = np.busday_count(P.ep.fillna(pd.Timestamp("2000-01-03")).values.astype("datetime64[D]"), P.date.values.astype("datetime64[D]"))
        P["dse"] = np.where(dse <= 120, dse, 999).astype(float)
    return P.sort_values("date").reset_index(drop=True)

def dumps(Pall, min_vol=None, min_disc=None, max_disc=None):
    min_vol, min_disc, max_disc = min_vol or MIN_VOL, min_disc or MIN_DISC, max_disc or MAX_DISC
    return Pall[(Pall.vol >= min_vol) & Pall.disc.between(min_disc, max_disc)].copy()

t0 = time.time(); Pall = build(); P0 = dumps(Pall)
print(f"{len(Pall):,} loose dumps; {len(P0):,} with the default definition ({time.time() - t0:.0f}s)")
print(f"default dumps: hit {P0.y.mean():.1%}, mean hedged return {P0.ret.mean() * 1e4:+.0f} bps, median {P0.ret.median() * 1e4:+.0f} bps "
      f"(before any model, all dumps from {P0.date.min().date()})")

**What the raw dumps look like before any model.** Below: the average hedged return by discount size, by indexer, by hour of the trade, and by days since a stress episode started. These are the patterns the model learns, shown on the whole sample, so in-sample. The model in § 7 never sees a quarter before predicting it.

In [ ]:
d = P0.copy()
d["disc_bucket"] = pd.cut(d.disc, [50, 75, 100, 150, 250, 500, 2000])
d["hour_bucket"] = pd.cut(d.hour, [0, 11, 13, 15, 16, 17, 24], labels=["<11h", "11-13h", "13-15h", "15-16h", "16-17h", ">=17h"])
d["dse_bucket"] = pd.cut(d.dse.replace(999, 500), [-1, 20, 40, 90, 120, 1000], labels=["0-20 bd", "20-40", "40-90", "90-120", "no episode"]) if "dse" in d else None
fig, axs = plt.subplots(1, 4, figsize=(16, 3.4), dpi=100)
for ax, col, title in zip(axs, ["disc_bucket", "indexer", "hour_bucket", "dse_bucket"], ["discount (bps)", "indexer", "time of trade", "days since stress start"]):
    g = d.groupby(col, observed=True).ret.agg(["mean", "size"])
    ax.bar(range(len(g)), g["mean"] * 1e4, color="#2a78d6")
    ax.set_xticks(range(len(g))); ax.set_xticklabels([str(x) for x in g.index], rotation=30, fontsize=7)
    for i, n in enumerate(g["size"]): ax.text(i, 0, f"n={n}", ha="center", va="bottom", fontsize=6.5, color="w" if g["mean"].iloc[i] > 0 else "k")
    ax.set_title(f"mean hedged return by {title} (bps)", fontsize=8.5, loc="left"); ax.axhline(0, color="k", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 7. The model (walk-forward LightGBM)

For each quarter from `OOS_START`:
1. **Train** a LightGBM classifier (gradient-boosted trees, kept small and regularised) on every dump whose **exit** happened at least 5 bd before the quarter starts. The label `y` is whether the hedged return was positive.
2. **Predict** the quarter's dumps.
3. **Calibrate** the raw scores into probabilities with isotonic regression, fitted on earlier **out-of-sample** predictions whose outcome is known. This maps the score to the hit rate it actually delivered. Then "p ≥ 0.70" means "historically ~70% of such dumps made money".

Nothing from the test quarter or later is used anywhere in the fit.

In [ ]:
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score

def walk_forward(P, feats=None, params=None):
    feats = [f for f in (feats or FEATS) if f in P]; params = params or LGB_PARAMS
    P = P.drop(columns=["p", "pc"], errors="ignore").copy(); P["p"] = np.nan
    q_of = P.date.dt.to_period("Q")
    for q in pd.period_range(OOS_START, P.date.max(), freq="Q"):
        test = q_of == q; train = P.te < q.start_time - pd.offsets.BDay(5)
        if test.sum() == 0 or train.sum() < 2000: continue
        m = lgb.LGBMClassifier(**params).fit(P.loc[train, feats], P.loc[train, "y"])
        P.loc[test, "p"] = m.predict_proba(P.loc[test, feats])[:, 1]
    P["pc"] = np.nan
    for q in sorted(q_of.unique()):
        past = P[(P.te < q.start_time) & P.p.notna()]
        if len(past) >= 1000:
            cur = q_of == q; P.loc[cur, "pc"] = IsotonicRegression(out_of_bounds="clip").fit(past.p, past.y).predict(P.loc[cur, "p"])
    return P

P = walk_forward(P0)
d = P.dropna(subset=["pc"])
print(f"out-of-sample dumps with a calibrated p: {len(d):,} (from {d.date.min().date()}) | AUC {roc_auc_score(d.y, d.pc):.3f}")
print(f"all dumps: hit {d.y.mean():.1%}, mean {d.ret.mean() * 1e4:+.0f} bps | p >= {CUT}: {int((d.pc >= CUT).sum()):,} dumps, "
      f"hit {d[d.pc >= CUT].y.mean():.1%}, mean {d[d.pc >= CUT].ret.mean() * 1e4:+.0f} bps")
cal = d.groupby(pd.cut(d.pc, [0, .5, .6, .7, .8, .9, 1]), observed=True).agg(n=("y", "size"), predicted=("pc", "mean"), realised=("y", "mean"),
                                                                             mean_bps=("ret", lambda s: s.mean() * 1e4))
print("calibration (out of sample): predicted vs realised hit rate"); display(cal.round(3))

## 8. The fund simulation (book)

The model gives a list of good dumps. The book turns it into a fund:
1. **Selection.** Keep dumps with p ≥ `CUT` and no payment failure of the bond or issuer in the last 90 days.
2. **Capture.** Only `CAPTURE` of each day's dumps reach us. Shuffle with a seed and keep every 5th dump of each day (20%). Results depend on which ones are drawn, so every number is an average over `N_DRAWS` seeds.
3. **Size.** Size = the smallest of:
   - the dump size;
   - the bond cap (4% of capital, or 1% if the bond or the issuer's worst bond is below 90% of par);
   - what is left under the 15% issuer cap;
   - free capital.

   Positions below R$ 100k are skipped.
4. **P&L.** Booked on the exit date: size × hedged return. Idle cash earns CDI, so the fund's equity = capital grown at CDI + cumulative excess P&L.

`stats()` reports, as averages over the draws:
- **excess**: return over CDI per year;
- **Sharpe** of monthly excess returns;
- **worst month**;
- **max drawdown relative to CDI**;
- **trades** and **hit rate**;
- **capital used**: average share of capital in positions.

In [ ]:
def book(P, seed, capital=None, capture=None, cut=None, cap_bond=None, cap_block="cfg", cap_distress=None, distress_pct=None, cost=None):
    capital = capital or CAPITAL; capture = capture or CAPTURE; cut = CUT if cut is None else cut
    cap_bond = cap_bond or CAP_BOND; cap_block = CAP_BLOCK if cap_block == "cfg" else cap_block
    cap_distress = CAP_DISTRESS if cap_distress is None else cap_distress; distress_pct = distress_pct or DISTRESS_PCT
    X = P[(P.pc >= cut) & (P.fail_bond90 == 0) & (P.fail_iss90 == 0)]
    if cost is not None: X = X.assign(ret=X.ret + 2 * (COST_BPS - cost) / 1e4)
    X = X.sample(frac=1, random_state=seed)
    X = X[X.groupby("date").cumcount() % int(round(1 / capture)) == 0].sort_values("date", kind="stable")
    held, rows = [], []
    for r in X.itertuples():
        held = [h for h in held if h[0] > r.date]                    # positions still open
        used = sum(h[1] for h in held); iss = sum(h[1] for h in held if h[2] == r.root)
        cap = cap_block if (cap_block and r.vol >= BLOCK_MIN) else cap_bond
        if r.fair < distress_pct or (r.iss_min == r.iss_min and r.iss_min < distress_pct): cap = min(cap, cap_distress)
        size = min(r.vol, cap * capital, CAP_ISSUER * capital - iss, capital - used)
        if size < 1e5: continue
        held.append((r.te, size, r.root)); rows.append((r.date, r.te, r.code, r.indexer, r.vol, size, r.ret))
    Tr = pd.DataFrame(rows, columns=["t", "te", "code", "indexer", "print_vol", "size", "ret"])
    days = pd.bdate_range(OOS_START, P.te.max())
    pnl = (Tr["size"] * Tr.ret).groupby(Tr.te).sum().reindex(days, fill_value=0).cumsum()
    cdi_eq = capital * CDI.reindex(days).ffill() / CDI.asof(days[0])
    return Tr, days, cdi_eq + pnl, cdi_eq

def stats(P, draws=N_DRAWS, **kw):
    out = []
    for s in range(draws):
        Tr, days, eq, cdi_eq = book(P, s, **kw)
        cap = kw.get("capital") or CAPITAL; yrs = len(days) / 252
        m = (eq.resample("ME").last().pct_change() - cdi_eq.resample("ME").last().pct_change()).dropna()
        rel = eq / cdi_eq
        inv = pd.Series(0.0, index=days)
        for r in Tr.itertuples(): inv.loc[r.t:r.te] += r.size
        out.append(dict(excess=(eq.iloc[-1] / cap) ** (1 / yrs) - (cdi_eq.iloc[-1] / cap) ** (1 / yrs), sharpe=m.mean() / m.std() * np.sqrt(12),
                        worst_month=m.min(), mdd_vs_cdi=(rel / rel.cummax() - 1).min(), trades=len(Tr), hit=(Tr.ret > 0).mean(),
                        capital_used=inv.mean() / cap, by_year=rel.groupby(rel.index.year).apply(lambda z: z.iloc[-1] / z.iloc[0] - 1).to_dict(),
                        m=m, curve=(days, eq, cdi_eq), Tr=Tr))
    R = pd.DataFrame(out)
    summ = {k: R[k].mean() for k in ("excess", "sharpe", "worst_month", "mdd_vs_cdi", "trades", "hit", "capital_used")}
    summ["excess_p5"] = R.excess.quantile(0.05); summ["by_year"] = pd.DataFrame(list(R.by_year)).mean().to_dict()
    return summ, R

def show(s):
    return (f"CDI {s['excess']:+.2%}/yr (p5 {s['excess_p5']:+.2%}), Sharpe {s['sharpe']:.2f}, worst month {s['worst_month']:+.2%}, "
            f"max DD vs CDI {s['mdd_vs_cdi']:+.2%}, {s['trades']:.0f} trades, hit {s['hit']:.1%}, capital used {s['capital_used']:.0%}")

## 9. Backtest and equity curve

The default fund: R$ 50 mi, 20% capture, 1 bp per side, cash in CDI, 30 random capture draws. The curve shown is the median draw.

**Benchmarks**
- **CDI**: what idle cash earns.
- **Passive CDI+ debentures**: an equal-weight portfolio of every CDI+ debenture, marked daily at ANBIMA, earning the change in mark + CDI + its spread. This is roughly what a passive credit fund earns before fees.

In [ ]:
def passive_cdi_plus(days, capital):
    A, B = DATA["A"], DATA["B"]
    A = A[A.code.map(B.indexer) == "CDI+"].sort_values(["code", "date"]).copy()
    A["r"] = A.groupby("code").pct_par.pct_change(fill_method=None); A = A[A.r.abs() < 0.05]
    A["tot"] = (1 + A.r) * (1 + A.date.map(cdi_d).fillna(0)) * (1 + A.code.map(B.spread).fillna(0).clip(0, 10) / 100) ** (1 / 252) - 1
    return (1 + A.groupby("date").tot.mean().reindex(days).fillna(0)).cumprod() * capital

BASE, RB = stats(P, draws=30)
print("DEFAULT:", show(BASE))
print("by year (vs CDI):", {k: f"{v:+.2%}" for k, v in BASE["by_year"].items()})
med = RB.sort_values("excess").iloc[len(RB) // 2]
days, eq, cdi_eq = med.curve
pas = passive_cdi_plus(days, CAPITAL)
med.Tr.to_csv(OUT / "trades_median_draw.csv", index=False)
yrs = len(days) / 252; ann = lambda s: (s.iloc[-1] / s.iloc[0]) ** (1 / yrs) - 1
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(11, 6.5), dpi=110, gridspec_kw={"height_ratios": [2.4, 1]}, sharex=True)
ax.plot(days, eq / CAPITAL * 100, lw=2.2, color="#1baf7a", label=f"Strategy {ann(eq):.1%}/yr (CDI {BASE['excess']:+.1%}, Sharpe {BASE['sharpe']:.1f})")
ax.plot(days, pas / CAPITAL * 100, lw=1.3, color="#d97a29", label=f"Passive CDI+ debentures {ann(pas):.1%}/yr")
ax.plot(days, cdi_eq / CAPITAL * 100, lw=1.2, ls="--", color="k", label=f"CDI {ann(cdi_eq):.1%}/yr")
ax.set_ylabel("base 100"); ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title(f"{COST_BPS:g} bp per side, cash in CDI, R$ {CAPITAL / 1e6:.0f} mi, capture {CAPTURE:.0%}, median of 30 draws", loc="left", fontsize=9)
rel = eq / cdi_eq; ax2.fill_between(days, (rel / rel.cummax() - 1) * 100, 0, color="#e34948", alpha=0.35); ax2.set_ylabel("DD vs CDI %")
for a in (ax, ax2): a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(OUT / "equity_curve.png"); plt.show()
print(f"median draw: {len(med.Tr)} trades, mean {med.Tr.ret.mean() * 1e4:+.0f} bps, hit {(med.Tr.ret > 0).mean():.0%}, "
      f"by indexer: {med.Tr.groupby('indexer').ret.agg(lambda s: f'{len(s)} @ {s.mean() * 1e4:+.0f} bps').to_dict()}")

## 10. Robustness analysis

Each variant changes **one** thing from the default and reports the mean over capture draws. A robust strategy stays positive across reasonable variants. If one choice dominates, it is a fragile point to confirm live.

| Block | What changes | Why |
|---|---|---|
| A | Book: cut, capture, cost, bond cap, distressed cap / threshold, fund size | Are the parameters cherry-picked? |
| B | Dump definition: min / max discount, min print size (model refit) | Is the result specific to one definition? |
| C | Holding period 10 / 42 bd | Is 21 bd lucky? |
| D | **Lag test**: fair value and state features from t − 2 instead of t − 1 | If results collapse, the model was using stale-mark artifacts |
| E | Model: tree settings, seeds, feature subsets, **no model at all** | What does the model add over "buy every dump"? |
| F | Time: by half and by year | Is the edge stable or decaying? |
| G | Bootstrap of monthly excess returns | How likely is the result to be luck? |

In [ ]:
rows = []
def add(group, name, s, default=False):
    rows.append(dict(group=group, variant=name, default=default, **{k: v for k, v in s.items() if k != "by_year"}))
    print(f"{group:<26} {name:<40} {show(s)}")

add("default", "default (30 draws)", BASE, True)
for v in (0.6, 0.65, 0.7, 0.75, 0.8): add("A cut", str(v), stats(P, cut=v)[0], v == CUT)
for v in (0.1, 0.2, 0.33, 0.5, 1.0): add("A capture", f"{v:.0%}", stats(P, capture=v)[0], v == CAPTURE)
for v in (0, 1, 3, 5, 10): add("A cost per side (bps)", str(v), stats(P, cost=v)[0], v == COST_BPS)
for v in (0.02, 0.04, 0.06): add("A bond cap", f"{v:.0%}", stats(P, cap_bond=v)[0], v == CAP_BOND)
for v in (0.0, 0.005, 0.01, 0.02, 0.04): add("A distressed cap", f"{v:.1%}" + (" (skip)" if v == 0 else " (no limit)" if v == 0.04 else ""), stats(P, cap_distress=v)[0], v == CAP_DISTRESS)
for v in (85, 90, 95): add("A distressed threshold", f"< {v}% par", stats(P, distress_pct=v)[0], v == DISTRESS_PCT)
for v in (10e6, 50e6, 200e6, 500e6): add("A fund size", f"R$ {v / 1e6:.0f} mi", stats(P, capital=v)[0], v == CAPITAL)
for v in (30, 50, 100, 200): add("B min discount (bps)", str(v), stats(walk_forward(dumps(Pall, min_disc=v)))[0], v == MIN_DISC)
for v in (500, 1000, 2000, 5000): add("B max discount (bps)", str(v), stats(walk_forward(dumps(Pall, max_disc=v)))[0], v == MAX_DISC)
for v in (1e5, 3e5, 1e6, 3e6): add("B min print size", f"R$ {v / 1e3:.0f}k", stats(walk_forward(dumps(Pall, min_vol=v)))[0], v == MIN_VOL)
for k, vals in (("num_leaves", (7, 31)), ("n_estimators", (150, 600)), ("min_child_samples", (100, 400))):
    for v in vals: add(f"E model {k}", str(v), stats(walk_forward(P0, params={**LGB_PARAMS, k: v}))[0])
for seed in (1, 2, 3, 4): add("E model seed", str(seed), stats(walk_forward(P0, params={**LGB_PARAMS, "random_state": seed}))[0])
for nm, drop in (("without fund stress (CVM)", ["dse"]), ("without intraday (time, n-th dump, lot)", ["hour", "n_prior_today", "round_lot"]),
                 ("without momentum / levels", ["mom5", "mom20", "iss_mom5", "iss_min", "fair"]),
                 ("only the print", [f for f in FEATS if f not in ("disc", "log_vol", "size_adv", "dur", "is_cdi", "is_ipca", "is_pre")])):
    add("E features", nm, stats(walk_forward(P0, feats=[f for f in FEATS if f not in drop]))[0])
add("E no model", "buy every dump (risk limits only)", stats(P.assign(pc=1.0))[0])
mm = pd.concat(list(RB.m), axis=1).mean(axis=1); h = len(mm) // 2
for nm, x in (("first half", mm.iloc[:h]), ("second half", mm.iloc[h:])):
    print(f"F {nm} {x.index[0]:%Y-%m}..{x.index[-1]:%Y-%m}: CDI {(1 + x).prod() ** (12 / len(x)) - 1:+.2%}/yr, Sharpe {x.mean() / x.std() * np.sqrt(12):.2f}")
print("F by year:", {k: f"{v:+.2%}" for k, v in BASE["by_year"].items()})
rng = np.random.default_rng(0); bs = [rng.choice(mm.values, len(mm)) for _ in range(5000)]
print(f"G bootstrap: excess p5 {np.percentile([(1 + b).prod() ** (12 / len(b)) - 1 for b in bs], 5):+.2%}/yr, "
      f"Sharpe p5 {np.percentile([b.mean() / b.std() * np.sqrt(12) for b in bs], 5):.2f}, P(lose to CDI) {np.mean([(1 + b).prod() < 1 for b in bs]):.2%}")
if RUN_FULL_ROBUSTNESS:
    for v in (10, 42): add("C holding period (bd)", str(v), stats(walk_forward(dumps(build(hold_bd=v))))[0])
    add("D lag test", "features from t-2", stats(walk_forward(dumps(build(lag_bd=2))))[0])
SENS = pd.DataFrame(rows); SENS.to_csv(OUT / "sensitivity.csv", index=False)

In [ ]:
G_ = [g for g in dict.fromkeys(SENS.group) if g != "default"]
nr = int(np.ceil(len(G_) / 4))
fig, axs = plt.subplots(nr, 4, figsize=(16, 2.9 * nr), dpi=100); axs = axs.ravel()
for ax, g in zip(axs, G_):
    d = SENS[SENS.group == g]
    ax.barh(range(len(d)), d.excess * 100, color=["#1baf7a" if x else "#2a78d6" for x in d.default])
    ax.errorbar(d.excess_p5 * 100, range(len(d)), fmt="|", color="k")
    for i, (e, sh) in enumerate(zip(d.excess, d.sharpe)): ax.text(max(e * 100, 0) + 0.2, i, f"Sh {sh:.1f}", va="center", fontsize=7)
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d.variant, fontsize=7); ax.axvline(0, color="k", lw=0.8)
    ax.set_title(g, fontsize=8.5, loc="left"); ax.spines[["top", "right"]].set_visible(False)
for ax in axs[len(G_):]: ax.axis("off")
fig.suptitle("Excess over CDI per year (%) - green = default, | = 5th percentile of capture draws", x=0.01, ha="left", fontsize=10)
plt.tight_layout(); plt.savefig(OUT / "sensitivity.png"); plt.show()
print(f"variants positive: {(SENS.excess > 0).sum()} of {len(SENS)} | min {SENS.excess.min():+.2%} ({SENS.loc[SENS.excess.idxmin(), 'group']} "
      f"{SENS.loc[SENS.excess.idxmin(), 'variant']}) | max {SENS.excess.max():+.2%}")

## 11. Scaling

**The constraint is flow, not capital.** We can only buy what sellers dump. Average inventory ≈ monthly dump volume we capture × the holding period. The cells below measure:
1. how much model-approved dump volume the market produces each month;
2. where the return sits by lot size;
3. a **block sleeve**: a separate model for dumps ≥ R$ 3 mi plus an 8% cap for them, across fund size × capture.

**Levers, in order of impact**
1. **Capture more flow.** Be on dealers' RFQ lists and the electronic platforms; see the flow the house's own desks see; cross-trade with house funds facing redemptions (within compliance rules on price and cross-trades).
2. **Large blocks.** If dumps above R$ 10 mi carry most of the volume and still revert, they are the scalable pool.
3. **Exit into buyers** (funds with inflows, distribution platforms) instead of waiting 21 bd. Most of the reversion happens in ~10 bd (§ 10, block C).
4. **More instruments with the same mechanics:** CRI, CRA, Letras Financeiras, FIDC seniors. This needs marks for them.
5. **Capital efficiency:** hedge with DI1 / DAP futures instead of Tesouro, and fund inventory in repo.
6. **Lean into stress windows:** margins are highest 40-90 bd after a credit-fund stress episode begins.

**Warnings**
- If the edge has shrunk recently (§ 10, block F), find out whether that is competition (permanent) or credit stress (temporary) before scaling.
- At high capture we would set the dump prices ourselves, and the discount shrinks. Only live trading measures this.

In [ ]:
Q = P[(P.date >= OOS_START) & (P.pc >= CUT)].copy()
mv = Q.groupby(Q.date.dt.to_period("M")).vol.sum() / 1e6
print(f"model-approved dump volume: median R$ {mv.median():,.0f} mi / month (p25 {mv.quantile(.25):,.0f}, p75 {mv.quantile(.75):,.0f})")
for cap_ in (0.2, 0.5, 1.0): print(f"  capture {cap_:.0%}: average inventory ~ R$ {mv.median() * cap_ * HOLD_BD / 21:,.0f} mi (hold {HOLD_BD} bd)")
Q["bucket"] = pd.cut(Q.vol, [0, 1e6, 3e6, 1e7, 1e12], labels=["< 1 mi", "1-3 mi", "3-10 mi", "> 10 mi"])
tab = Q.groupby("bucket", observed=True).agg(dumps=("ret", "size"), volume_share=("vol", "sum"), mean_bps=("ret", lambda s: s.mean() * 1e4), hit=("y", "mean"))
tab["volume_share"] = tab.volume_share / tab.volume_share.sum(); display(tab.round(3))
print("measured round-trip dealer markup by size (bps): <100k 178 | 100-300k 146 | 300k-1m 60 | 1-3m 20 | 3-10m 10 | >10m 7")

In [ ]:
Pblk = walk_forward(P0[P0.vol >= BLOCK_MIN]); Psml = walk_forward(P0[P0.vol < BLOCK_MIN])
Psplit = pd.concat([Pblk, Psml]).sort_values("date").reset_index(drop=True)
scal = []
for capital in (50e6, 200e6, 500e6, 1000e6):
    for capture in (0.2, 0.5):
        for nm, PP, cb in (("default (one model, 4% cap)", P, None), ("block sleeve (separate model, 8% cap for >= 3 mi)", Psplit, 0.08)):
            s = stats(PP, draws=5, capital=capital, capture=capture, cap_block=cb)[0]
            scal.append(dict(capital=capital / 1e6, capture=capture, version=nm, excess=s["excess"], sharpe=s["sharpe"], worst=s["worst_month"],
                             capital_used=s["capital_used"], pnl_mi_per_yr=s["excess"] * capital / 1e6))
            print(f"R$ {capital / 1e6:>5.0f} mi | capture {capture:.0%} | {nm:<50} {show(s)}")
SCAL = pd.DataFrame(scal); SCAL.to_csv(OUT / "scaling.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 4), dpi=100)
for (nm, cap_), d in SCAL.groupby(["version", "capture"]):
    ax[0].plot(d.capital, d.excess * 100, marker="o", label=f"{nm[:22]}, capture {cap_:.0%}")
    ax[1].plot(d.capital, d.pnl_mi_per_yr, marker="o", label=f"{nm[:22]}, capture {cap_:.0%}")
for a, yl in zip(ax, ("excess over CDI, %/yr", "excess P&L, R$ mi / yr")):
    a.set_xscale("log"); a.set_xlabel("fund size, R$ mi"); a.set_ylabel(yl); a.spines[["top", "right"]].set_visible(False)
ax[0].legend(fontsize=7, frameon=False); plt.tight_layout(); plt.savefig(OUT / "scaling.png"); plt.show()

## 12. Live test: what data to request, daily routine, go / no-go

### 12.1 Data to request (internal SQL / desk systems)
Every query is date-limited: one month per query for history, D−1 for the daily refresh.

| # | Data | Table (known) / what to ask for | Columns | History | Refresh | Check |
|---|---|---|---|---|---|---|
| S1 | **B3 secondary trades** (debentures; for scaling also CRI, CRA, LF) | `B3_CREDITO_OPERACOES` | `datanegocio`, **time of trade** (`horanegocio` or equivalent: confirm), `codigoif`, `instrumentofinanceiro`, `qtdadenegociada`, `preconegocio`, `volumefinanceiro`, `taxanegocio`, `origemnegocio`, status / cancellation flag, trade id | 2023-04 → | Daily (D−1) | No duplicates; vol = qty × PU; cancelled removed; time present |
| S2 | **ANBIMA marks**, all debentures incl. matured / defaulted | `tbhh3003_anbima_debenture_mercado_secundario` | `data_referencia`, `codigo_ativo`, `grupo`, `taxa_indicativa`, `pu`, `percent_pu_par`, `duration` (business days?) | 2022 → | Daily | PU / %PU par smooth; duration unit |
| S3 | ANBIMA marks for CRI / CRA (scaling) | Ask | As S2 | 2023 → | Daily | Only to add those instruments |
| S4 | **Bond master** | Ask (or keep the SND download) | Code, ISIN, issuer CNPJ, indexer, spread / coupon, dates, call clause, incentivized flag, economic group | Current + dead | Weekly | Every traded code mapped |
| S5 | **Payment events and defaults** | Ask (or keep SND) | Code, date, event type, status | 2014 → | Daily | Failures flagged the same day |
| S6 | **Rates curves** | Ask: DI1 / DAP / NTN-B curves | Date, vertex, rate | 2022 → | Daily | The instruments the hedge will actually use |
| S7 | CDI, IPCA | Any market-data table (or BCB) | Date, value | 2018 → | Daily / monthly | |
| S8 | **House fund positions** | `VW_FUNDOS_POSICAO_EXPLODIDA_DOWN` | `date`, `codigo_IAM_fundo`, `codigo_IAM_fundo_origem`, `codigo_IAM_ativo`, `tipo`, `perc_nav`, `isin`, `issuer` | 2021 → | Daily | Which house funds hold each bond → internal crosses, redemption pressure |
| S9 | **House fund flows / redemption queue** | Ask | Fund, date, requested redemption, payment date, NAV | 2023 → | Daily | Forced-seller flow **before** it reaches the market |
| S10 | **Desk RFQ log** (most important new data) | Ask | Timestamp, code, side, size, price / rate asked, counterparty type, our quote, traded (Y/N), traded price, number of competing dealers | As kept | Real time | Measures **capture** and real **fill cost**, which the backtest cannot test |
| S11 | Desk executions | Ask | Timestamp, code, side, size, price, counterparty, fees | Live period | Real time | Real cost per lot size vs 1 bp |

### 12.2 Daily routine (paper trading first, 3 months)
1. **07:00.** Re-run §§ 4-6 with `DOWNLOAD = True` to collect D−1 data.
2. **Quarterly** (or after a data fix), retrain with `train_final()` (§ 13).
3. **Before the open**, run `bid_sheet()`. For each bond it shows yesterday's fair value, the discount at which a standard R$ 1 mi dump would be bought, the max size and the flags.
4. **During the day**, for each RFQ / dump seen, run `score_rfq(code, price, size, time, n_prior_today)` and record it with `log_rfq(...)`, including the ones we do not bid.
5. **Evening:** `update_paper_log()` fills exits that have arrived.
6. **Monthly:** `paper_report()` against the criteria below.

### 12.3 Pre-registered go / no-go (fixed before the test)
After 3 months of paper trading with ≥ 60 paper fills:
- realised hit rate ≥ 65% and mean ≥ +40 bps per trade **after real fill cost** (S11);
- measured capture ≥ 10% of model-approved dumps;
- real round-trip cost at R$ 300k-1 mi ≤ 30 bps;
- no month worse than −2% of the paper book;
- the 21 bd and 10 bd exits run side by side, and the one that wins **in paper** is chosen.

Go live small (R$ 10-50 mi) only if **all** pass. Stop if 3 consecutive months are below CDI.

## 13. Live tools: final model, bid sheet, RFQ scorer, paper log

- **`train_final()`** trains on every dump whose exit is known. The probability calibration comes from the walk-forward out-of-sample predictions above. It is saved to `notebook_out/final_model.joblib`.
- **`bond_state()`** has, per bond, everything known at the open: yesterday's mark, duration, momentum, issuer state, ADV, last trade, payment failures and the stress counter.
- **`score_rfq()`** prices one offer. **`bid_sheet()`** gives, for each bond, the smallest discount at which a standard dump would be bought.

In [ ]:
import joblib

def train_final(P, asof=None):
    asof = pd.Timestamp(asof or DATA["T"].date.max())
    feats = [f for f in FEATS if f in P]
    tr = P[P.te < asof - pd.offsets.BDay(5)]
    m = lgb.LGBMClassifier(**LGB_PARAMS).fit(tr[feats], tr.y)
    oos = P.dropna(subset=["p"]); iso = IsotonicRegression(out_of_bounds="clip").fit(oos.p, oos.y)
    joblib.dump({"model": m, "iso": iso, "feats": feats, "asof": asof,
                 "config": {k: globals()[k] for k in ("CUT", "MIN_DISC", "MAX_DISC", "HOLD_BD", "CAP_BOND", "CAP_ISSUER", "CAP_DISTRESS", "DISTRESS_PCT")}},
                OUT / "final_model.joblib")
    print(f"final model trained on {len(tr):,} closed dumps up to {asof.date()} -> {OUT / 'final_model.joblib'}")
    return {"model": m, "iso": iso, "feats": feats}

def bond_state(asof=None):
    asof = pd.Timestamp(asof or (DATA["T"].date.max() + pd.offsets.BDay(1))); y = asof - pd.offsets.BDay(1)
    A, B, T, E = DATA["A"], DATA["B"], DATA["T"], DATA["E"]
    A = A[A.date <= y].sort_values(["code", "date"]).copy()
    A["mom5"] = A.pct_par - A.groupby("code").pct_par.shift(5); A["mom20"] = A.pct_par - A.groupby("code").pct_par.shift(20)
    last = A.groupby("code").tail(1).set_index("code"); last = last[last.date >= y - pd.Timedelta(days=7)]
    St = pd.DataFrame({"fair": last.pct_par, "pu_par_last": last.pu_par, "dur": last.dur, "mom5": last.mom5, "mom20": last.mom20})
    St["indexer"], St["root"] = St.index.map(B.indexer), St.index.map(B.root)
    St = St[St.indexer.isin(["CDI+", "%CDI", "IPCA+", "PRE"])]
    St = St.join(last.assign(root=last.index.map(B.root)).groupby("root").agg(iss_mom5=("mom5", "median"), iss_min=("pct_par", "min")), on="root")
    Ty = T[(T.date <= y) & (T.date > y - pd.Timedelta(days=95))]
    dv = Ty.groupby(["code", "date"]).vol.sum().reset_index()
    St["adv63"] = dv[dv.date > y - pd.offsets.BDay(63)].groupby("code").vol.sum().reindex(St.index) / 63
    St["last_trade"] = T[T.date <= y].groupby("code").date.max().reindex(St.index)
    F = E[E.failed & (E.date < asof) & (E.date >= asof - pd.Timedelta(days=90))]
    St["fail_bond90"] = St.index.isin(F.code).astype(float); St["fail_iss90"] = St.root.isin(F.code.map(B.root)).astype(float)
    st = DATA["stress"]; ep = st[st <= asof].max() if st is not None and (st <= asof).any() else None
    dse = float(np.busday_count(ep.date(), asof.date())) if ep is not None else 999.0
    St["dse"] = dse if dse <= 120 else 999.0
    for k, nm in (("CDI+", "is_cdi"), ("IPCA+", "is_ipca"), ("PRE", "is_pre")): St[nm] = (St.indexer == k).astype(float)
    St["state_date"] = asof
    return St

def _features(St, code_, pct, size, hhmm="16:30", n_prior_today=0, qty=None):
    r = St.loc[code_]
    return pd.DataFrame([{"disc": (r.fair - pct) / r.fair * 1e4, "log_vol": np.log(size), "size_adv": size / r.adv63 if r.adv63 else np.nan,
                          "dur": r.dur, "fair": r.fair, "is_cdi": r.is_cdi, "is_ipca": r.is_ipca, "is_pre": r.is_pre,
                          "days_since": (r.state_date - r.last_trade).days if pd.notna(r.last_trade) else np.nan, "mom5": r.mom5, "mom20": r.mom20,
                          "iss_mom5": r.iss_mom5, "iss_min": r.iss_min, "fail_bond90": r.fail_bond90, "fail_iss90": r.fail_iss90,
                          "hour": int(hhmm[:2]) + int(hhmm[3:5]) / 60, "n_prior_today": n_prior_today,
                          "round_lot": float(qty % 10 == 0) if qty else 1.0, "dse": r.dse}])

def max_size(St, code_, capital=None):
    r = St.loc[code_]
    return (CAP_DISTRESS if (r.fair < DISTRESS_PCT or (pd.notna(r.iss_min) and r.iss_min < DISTRESS_PCT)) else CAP_BOND) * (capital or CAPITAL)

def score_rfq(code_, price, size, hhmm="16:30", n_prior_today=0, qty=None, St=None, price_is_pu=False):
    """Score one incoming dump / RFQ. price = %PU par (or PU with price_is_pu=True)."""
    St = St if St is not None else STATE
    if code_ not in St.index: return {"code": code_, "decision": "NO - no mark yesterday"}
    pct = price / St.loc[code_, "pu_par_last"] * 100 if price_is_pu else price
    X = _features(St, code_, pct, size, hhmm, n_prior_today, qty)
    p = float(FINAL["iso"].predict(FINAL["model"].predict_proba(X[FINAL["feats"]])[:, 1])[0])
    disc = float(X.disc.iloc[0]); r = St.loc[code_]
    ok = (p >= CUT) and MIN_DISC <= disc <= MAX_DISC and r.fail_bond90 == 0 and r.fail_iss90 == 0
    return {"code": code_, "fair_pct": round(r.fair, 3), "price_pct": round(pct, 3), "disc_bps": round(disc, 1), "p": round(p, 3),
            "decision": "BUY" if ok else "NO", "max_size": round(min(size, max_size(St, code_)), 0), "indexer": r.indexer,
            "hedge": "short Tesouro x beta" if r.indexer in ("IPCA+", "PRE") else "none"}

def bid_sheet(size=1e6, hhmm="16:30", grid=(50, 75, 100, 150, 200, 300, 400, 500, 750, 1000), St=None):
    St = St if St is not None else STATE; out = []
    for code_ in St.index:
        r = St.loc[code_]
        if r.fail_bond90 or r.fail_iss90 or not np.isfinite(r.fair): continue
        Xg = pd.concat([_features(St, code_, r.fair * (1 - g / 1e4), size, hhmm) for g in grid], ignore_index=True)
        p = FINAL["iso"].predict(FINAL["model"].predict_proba(Xg[FINAL["feats"]])[:, 1]); ok = np.where(p >= CUT)[0]
        g = grid[ok[0]] if len(ok) else np.nan
        out.append({"code": code_, "indexer": r.indexer, "fair_pct": round(r.fair, 3), "bid_disc_bps": g,
                    "bid_pct": round(r.fair * (1 - g / 1e4), 3) if len(ok) else np.nan, "p_at_bid": round(float(p[ok[0]]), 3) if len(ok) else np.nan,
                    "best_p": round(float(p.max()), 3), "best_p_disc_bps": grid[int(p.argmax())], "max_size": max_size(St, code_),
                    "distressed": r.fair < DISTRESS_PCT, "stress_days": r.dse if r.dse < 999 else None})
    return pd.DataFrame(out).sort_values(["bid_disc_bps", "best_p"], ascending=[True, False])

LOG = OUT / "paper_trade_log.csv"
LOG_COLS = ["logged_at", "date", "time", "code", "side", "size", "price_pct", "fair_pct", "disc_bps", "p", "decision", "our_bid_pct", "filled",
            "fill_price_pct", "counterparty_type", "n_competing", "exit_date", "exit_pct", "ret_hedged", "notes"]

def log_rfq(code_, date, hhmm, size, price_pct, decision, p=None, fair_pct=None, our_bid_pct=None, filled=None, fill_price_pct=None,
            counterparty_type=None, n_competing=None, notes=""):
    """Append one RFQ / dump to the paper log (log the ones we do NOT bid too: they measure what we skip)."""
    row = dict(logged_at=pd.Timestamp.now().isoformat(timespec="seconds"), date=str(pd.Timestamp(date).date()), time=hhmm, code=code_, side="buy",
               size=size, price_pct=price_pct, fair_pct=fair_pct, disc_bps=(fair_pct - price_pct) / fair_pct * 1e4 if fair_pct else None, p=p,
               decision=decision, our_bid_pct=our_bid_pct, filled=filled, fill_price_pct=fill_price_pct, counterparty_type=counterparty_type,
               n_competing=n_competing, exit_date=None, exit_pct=None, ret_hedged=None, notes=notes)
    pd.DataFrame([row], columns=LOG_COLS).to_csv(LOG, mode="a", header=not LOG.exists(), index=False)

def update_paper_log():
    """Fill exits of filled paper trades: first trade day after date + HOLD_BD, daily VWAP in %PU par."""
    if not LOG.exists(): return None
    L = pd.read_csv(LOG); T = DATA["T"]; A = DATA["A"]
    Dl = T.merge(A[["code", "date", "pu_par"]], on=["code", "date"]).assign(pct=lambda d: d.pu / d.pu_par * 100)
    Dl = Dl.assign(pv=Dl.pct * Dl.vol).groupby(["code", "date"]).agg(pv=("pv", "sum"), vol=("vol", "sum")).reset_index()
    Dl["pct"] = Dl.pv / Dl.vol
    for i, r in L[(L.filled == True) & L.exit_date.isna()].iterrows():
        d = Dl[(Dl.code == r.code) & (Dl.date > pd.Timestamp(r.date) + pd.offsets.BDay(HOLD_BD))].sort_values("date")
        if len(d):
            L.loc[i, "exit_date"], L.loc[i, "exit_pct"] = str(d.date.iloc[0].date()), d.pct.iloc[0]
            L.loc[i, "ret_hedged"] = d.pct.iloc[0] / r.fill_price_pct - 1     # %PU par change; add the desk's hedge P&L separately
    L.to_csv(LOG, index=False); return L

def paper_report():
    if not LOG.exists(): print("no paper trades yet"); return
    L = pd.read_csv(LOG); f = L[L.filled == True]; c = f.dropna(subset=["ret_hedged"])
    print(f"RFQs logged {len(L)}, model BUY {int((L.decision == 'BUY').sum())}, filled {len(f)} "
          f"(capture of BUY = {len(f) / max((L.decision == 'BUY').sum(), 1):.0%}), closed {len(c)}")
    if len(c): print(f"closed: hit {(c.ret_hedged > 0).mean():.0%}, mean {c.ret_hedged.mean() * 1e4:+.0f} bps, worst {c.ret_hedged.min() * 1e4:+.0f} bps")

FINAL = train_final(P)
STATE = bond_state()
print("bond state as of", STATE.state_date.iloc[0].date(), "|", len(STATE), "bonds with a mark")
sheet = bid_sheet(); sheet.to_csv(OUT / "bid_sheet.csv", index=False)
n_ok = int(sheet.bid_disc_bps.notna().sum())
print(f"bid sheet: {n_ok} of {len(sheet)} bonds would be bought at some discount <= 1000 bps (p >= {CUT}); "
      f"stress days today: {STATE.dse.iloc[0] if STATE.dse.iloc[0] < 999 else 'no episode'}")
if n_ok == 0: print("(none passing = the model says do not bid today, e.g. early in a credit-fund stress episode)")
display(sheet.head(15))
ex_ = sheet.code.iloc[0] if len(sheet) else STATE.index[0]
print("example RFQ:", score_rfq(ex_, STATE.loc[ex_, "fair"] * (1 - 150 / 1e4), 1e6, "16:40"))

### Files written to `notebook_out/`
- `equity_curve.png`, `trades_median_draw.csv`: backtest
- `sensitivity.csv`, `sensitivity.png`: robustness
- `scaling.csv`, `scaling.png`: scaling
- `final_model.joblib`: live model and its configuration
- `bid_sheet.csv`: today's bid levels
- `paper_trade_log.csv`: the paper-trading log (`log_rfq` → `update_paper_log` → `paper_report`)

### Raw data written to `data/raw/` (kept as received)
- `snd_caracteristicas_full.parquet`
- `b3_trades/YYYYMMDD.parquet`
- `anbima_hist/{code}.parquet`
- `anbima_daily/dbYYMMDD.txt`
- `alt/snd_eventos/YYYY.txt`
- `alt/tesouro/td.csv`
- `alt/sgs_*_nb.json`
- `cvm/*.zip`